Microsoft Windows [Version 10.0.19045.6466]
(c) Microsoft Corporation. All rights reserved.

C:\Users\Comsci and Crypto\Desktop\Sample\CF-2\ml_training\scripts>py 1preprocess.py
[PREPROCESS] Loaded 700 rows from data/raw/final_700_enriched.csv
[PREPROCESS] ✓ 700 rows saved → data/processed/metadata_clean.csv
[PREPROCESS] Label distribution:
[PREPROCESS]   Overstimulating: 235 (33.6%)
[PREPROCESS]   Educational: 230 (32.9%)
[PREPROCESS]   Neutral: 235 (33.6%)

[PREPROCESS] Expected split (train_nb.py will confirm):
[PREPROCESS]   Train (70%): 490  |  Test (30%): 210
[PREPROCESS] Next step: python train_nb.py

C:\Users\Comsci and Crypto\Desktop\Sample\CF-2\ml_training\scripts>py 2baseline.py
====================================================================
  BASELINE RESULTS — All Four Metrics  (test_210.csv, n=210)
====================================================================
  Baseline                    Accuracy   Precision   Recall       F1   Over_Rec
  --------------------------------------------------------------------
  B1  Random chance             35.71%      35.67%    35.73%    35.66%      40.00%
  B2  Majority ('Overstimulating')    33.33%      11.11%    33.33%    16.67%     100.00%
  B3  Keyword heuristic         49.52%      60.46%    49.32%    47.56%      27.14%
====================================================================

  NOTE — Macro averages weight all 3 classes equally.
  Over_Rec = Overstimulating recall (child safety metric).

====================================================================
  B3 KEYWORD HEURISTIC — Per-Class Detail
====================================================================
                 precision    recall  f1-score   support

    Educational       0.61      0.39      0.48        69
        Neutral       0.41      0.82      0.54        71
Overstimulating       0.79      0.27      0.40        70

       accuracy                           0.50       210
      macro avg       0.60      0.49      0.48       210
   weighted avg       0.60      0.50      0.48       210

  Confusion Matrix  (rows=Actual, cols=Predicted):
                           Educa   Neutr   Overs
             Educational      27      41       1
                 Neutral       9      58       4
         Overstimulating       8      43      19

====================================================================
  YOUR MODEL MUST BEAT (all four metrics over baseline):
  Accuracy  > 49.52%    (B3 keyword heuristic)
  Precision > 60.46%    (macro)
  Recall    > 49.32%    (macro)
  F1        > 47.56%    (macro)
  Over_Rec  > 27.14%    (Overstimulating recall)
====================================================================

-> Run: py 3modelselection.py

C:\Users\Comsci and Crypto\Desktop\Sample\CF-2\ml_training\scripts>py 3model_selection.py
Training samples loaded: 490  <- must be 490
Label dist: {'Educational': 161, 'Neutral': 164, 'Overstimulating': 165}

==================================================================================
  Model                     Acc%   Prec%    Rec%      F1    ±Acc    Gap%
==================================================================================
  Complement NB           79.59%   79.74%   79.55%  0.7937  +-3.35%   15.9%
  Multinomial NB          79.18%   79.40%   79.14%  0.7895  +-3.45%   16.0%
  Logistic Regression     78.78%   79.40%   78.76%  0.7883  +-3.79%   19.1%
  Linear SVM              78.37%   78.80%   78.34%  0.7833  +-3.12%   21.6%
  Random Forest           73.88%   74.83%   73.90%  0.7405  +-6.66%   26.1%
==================================================================================

======================================================================
  COMPLEMENT NB — Per-Fold Detail (all four metrics)
======================================================================
  Fold       Accuracy   Precision   Recall       F1
  ------------------------------------------------
  Fold 1        84.69%      84.77%    84.63%  0.8466
  Fold 2        75.51%      75.56%    75.51%  0.7543
  Fold 3        81.63%      82.11%    81.60%  0.8162
  Fold 4        76.53%      76.74%    76.39%  0.7594
  Fold 5        79.59%      79.51%    79.64%  0.7919
  ------------------------------------------------
  Mean         79.59%      79.74%    79.55%  0.7937
  ±Std     +-   3.35%  +-   3.39%  +- 3.36%  +-0.0347

======================================================================
  MODEL SELECTION DECISION GUIDE
======================================================================
  Highest CV Accuracy   : Complement NB
  Highest Macro F1      : Complement NB
  Highest Macro Recall  : Complement NB
  Smallest Overfit Gap  : Complement NB

  Selected: Complement NB
  Reason  : Best or competitive on all four metrics with smallest
            overfit gap. Designed for complement class estimation
            (Rennie et al. 2003) — ideal for imbalanced 3-class text.

-> Test set (test_210.csv) still SEALED.
-> Next: py 4tunealpha.py

C:\Users\Comsci and Crypto\Desktop\Sample\CF-2\ml_training\scripts>py 4a_tune_alpha.py
Training samples loaded: 490  <- must be 490

================================================================================
     Alpha      Acc%     ±Acc    Prec%     Rec%        F1
================================================================================
  alpha=0.05      78.16%  +-2.10%   78.58%   78.11%  0.7802
  alpha=0.1       78.16%  +-1.66%   78.56%   78.11%  0.7801
  alpha=0.15      78.98%  +-1.38%   79.46%   78.93%  0.7884
  alpha=0.2       79.39%  +-1.76%   79.95%   79.34%  0.7926
  alpha=0.3       78.98%  +-2.10%   79.48%   78.94%  0.7884
  alpha=0.5       79.80%  +-2.77%   80.07%   79.76%  0.7966
  alpha=1.0       79.59%  +-3.35%   79.74%   79.55%  0.7937
  alpha=1.5       80.00%  +-3.14%   80.17%   79.96%  0.7975  <- BEST
  alpha=2.0       79.39%  +-2.77%   79.56%   79.35%  0.7916
================================================================================

======================================================================
  BEST ALPHA = 1.5 — Per-Fold Detail
======================================================================
  Fold       Accuracy   Precision   Recall       F1
  ------------------------------------------------
  Fold 1        84.69%      84.77%    84.63%  0.8466
  Fold 2        77.55%      77.63%    77.56%  0.7751
  Fold 3        82.65%      83.27%    82.61%  0.8261
  Fold 4        76.53%      76.74%    76.39%  0.7594
  Fold 5        78.57%      78.43%    78.63%  0.7804
  ------------------------------------------------
  Mean         80.00%      80.17%    79.96%  0.7975
  ±Std     +-   3.14%  +-   3.22%  +- 3.13%  +-0.0331

======================================================================
  STEP 4 COMPLETE
======================================================================
  Best alpha : 1.5
  CV Acc     : 80.00%
  CV F1      : 0.7975

-> Open 4b_tune_tfidf.py and confirm: BEST_ALPHA = 1.5
-> Then run: py 4b_tune_tfidf.py
-> Test set (test_210.csv) still SEALED.

C:\Users\Comsci and Crypto\Desktop\Sample\CF-2\ml_training\scripts>py 4b_tune_tfidf.py
Training samples loaded: 490  <- must be 490
Alpha fixed at         : 1.5  (from Step 4)
Test set               : still SEALED

Generating Figure 1: Stratified K-Fold split visualization...
  Saved -> C:\Users\Comsci and Crypto\Desktop\Sample\CF-2\ml_training\scripts\..\outputs\figures\fig1_stratified_kfold_split.png
========================================================================================
    max_feat   min_df      Acc%     ±Acc    Prec%     Rec%        F1
========================================================================================
  mf=5,000    md=1        81.22%  +-3.45%   81.30%   81.22%  0.8101
  mf=5,000    md=2        80.00%  +-3.14%   80.17%   79.96%  0.7975
  mf=5,000    md=3        80.61%  +-3.23%   81.05%   80.61%  0.8050
  mf=5,000    md=5        76.33%  +-5.41%   76.84%   76.32%  0.7644

  mf=10,000   md=1        80.20%  +-2.78%   80.34%   80.20%  0.7994
  mf=10,000   md=2        80.00%  +-3.14%   80.17%   79.96%  0.7975
  mf=10,000   md=3        80.61%  +-3.23%   81.05%   80.61%  0.8050
  mf=10,000   md=5        76.33%  +-5.41%   76.84%   76.32%  0.7644

  mf=15,000   md=1        79.80%  +-2.53%   79.92%   79.79%  0.7951
  mf=15,000   md=2        80.00%  +-3.14%   80.17%   79.96%  0.7975
  mf=15,000   md=3        80.61%  +-3.23%   81.05%   80.61%  0.8050
  mf=15,000   md=5        76.33%  +-5.41%   76.84%   76.32%  0.7644

  mf=20,000   md=1        79.80%  +-2.53%   79.92%   79.79%  0.7951
  mf=20,000   md=2        80.00%  +-3.14%   80.17%   79.96%  0.7975
  mf=20,000   md=3        80.61%  +-3.23%   81.05%   80.61%  0.8050
  mf=20,000   md=5        76.33%  +-5.41%   76.84%   76.32%  0.7644
========================================================================================

  Best config: max_features=5,000, min_df=1
  CV Acc : 81.22%  +-3.45%
  CV F1  : 0.8101
  vs Step 4 baseline: 80.00%  (↑ 1.22pp)

Generating Figure 2: Grid search heatmap...
  Saved -> C:\Users\Comsci and Crypto\Desktop\Sample\CF-2\ml_training\scripts\..\outputs\figures\fig2_gridsearch_heatmap.png
Generating Figure 3: Accuracy vs stability error bar plot...
  Saved -> C:\Users\Comsci and Crypto\Desktop\Sample\CF-2\ml_training\scripts\..\outputs\figures\fig3_accuracy_errorbar.png
Generating Figure 4: Per-fold accuracy of best configuration...
  Saved -> C:\Users\Comsci and Crypto\Desktop\Sample\CF-2\ml_training\scripts\..\outputs\figures\fig4_best_config_folds.png

=================================================================
  STEP 4b COMPLETE — ALL FIGURES SAVED
=================================================================
  fig1  Stratified K-Fold split    -> fold balance transparency
  fig2  Grid search heatmap        -> accuracy + std across all configs
  fig3  Accuracy vs stability      -> error bar + F1 overlay vs baseline
  fig4  Per-fold best config       -> fold-by-fold stability proof

  Best config found:
    BEST_ALPHA        = 1.5
    BEST_MAX_FEATURES = 5000
    BEST_MIN_DF       = 1
    CV Acc            = 81.22%  +-3.45%
    CV F1             = 0.8101

-> Open 5final_eval.py and set:
     BEST_ALPHA        = 1.5
     BEST_MAX_FEATURES = 5000
     BEST_MIN_DF       = 1
-> Then run: py 5final_eval.py  (unseals test_210.csv — one time only)
-> Test set (test_210.csv) still SEALED.

C:\Users\Comsci and Crypto\Desktop\Sample\CF-2\ml_training\scripts>py 5final_eval.py
Train : 490  <- must be 490
Test  : 210   <- must be 210

==========================================================
  CV RESULTS (training pool only, n=490, test still sealed)
==========================================================
  Fold       Accuracy   Precision   Recall       F1
  --------------------------------------------------
  Fold 1        84.69%      84.61%    84.63%  0.8460
  Fold 2        77.55%      77.97%    77.59%  0.7752
  Fold 3        85.71%      85.95%    85.73%  0.8566
  Fold 4        80.61%      80.59%    80.52%  0.8038
  Fold 5        77.55%      77.36%    77.62%  0.7687
  --------------------------------------------------
  Mean         81.22%      81.30%    81.22%  0.8101
  Std    +-    3.45%  +-   3.45%  +-  3.42%  +-0.0358

==========================================================
  FINAL HOLDOUT RESULTS  (test_210.csv — unsealed once)
==========================================================
  Alpha              : 1.5
  max_features       : 5000
  min_df             : 1

  Train accuracy     : 97.76%
  Holdout accuracy   : 84.76%
  Overfit gap        : 12.99%  (train - holdout)

  ── Macro averages (unweighted mean across 3 classes) ──
  Macro Precision    : 84.76%  (0.8476)
  Macro Recall       : 84.77%  (0.8477)
  Macro F1           : 84.73%  (0.8473)

  ── Weighted averages (weighted by class support) ──
  Weighted Precision : 84.75%  (0.8475)
  Weighted Recall    : 84.76%  (0.8476)
  Weighted F1        : 84.72%  (0.8472)

  ── Child safety metrics ──
  Bi-decision acc    : 91.43%  (Overstimulating vs Safe)
  Overstimulating recall : 90.00%  (missed detections = 6/70)
  CV mean (5-fold)   : 81.22% +-3.45%

==========================================================
  CLASSIFICATION REPORT (holdout, per-class breakdown)
==========================================================
                 precision    recall  f1-score   support

    Educational     0.8507    0.8261    0.8382        69
        Neutral     0.8406    0.8169    0.8286        71
Overstimulating     0.8514    0.9000    0.8750        70

       accuracy                         0.8476       210
      macro avg     0.8476    0.8477    0.8473       210
   weighted avg     0.8475    0.8476    0.8472       210


==========================================================
  PER-CLASS METRICS TABLE
==========================================================
  Class                 Precision   Recall       F1   Support
  ----------------------------------------------------------
  Educational              85.07%    82.61%    83.82%       69
  Neutral                  84.06%    81.69%    82.86%       71
  Overstimulating          85.14%    90.00%    87.50%       70

==========================================================
  CONFUSION MATRIX (rows=actual, columns=predicted)
==========================================================
                             Educa     Neutr     Overs
             Educational        57         7         5
                 Neutral         7        58         6
         Overstimulating         3         4        63

==========================================================
  THESIS SUMMARY ROW — Complement NB  (paste into Chapter 5)
==========================================================
  Holdout Acc : 84.76%
  CV Acc      : 81.22% +-3.45%
  Overfit Gap : 12.99%
  Macro P     : 0.8476
  Macro R     : 0.8477
  Macro F1    : 0.8473
  Bi-Dec Acc  : 91.43%
  Over. Recall: 90.00%

-> These are your FINAL numbers.
-> Do NOT re-run with different parameters after this.
-> Paste results into 6generate_figures.py

C:\Users\Comsci and Crypto\Desktop\Sample\CF-2\ml_training\scripts>

Microsoft Windows [Version 10.0.19045.6466]
(c) Microsoft Corporation. All rights reserved.

C:\Users\Comsci and Crypto\Desktop\Sample\CF-2\ml_training\scripts>py 6train_nb.py
═══ Training Final Thesis Model (NB-Only) ═══
Loaded 490 training samples.
Loaded 210 test samples.
Vectorizing text...
Training ComplementNB...

=======================================================
NAÏVE BAYES HOLDOUT ACCURACY: 83.81%
=======================================================

[Standalone NB] Detailed Classification Report:
                 precision    recall  f1-score   support

    Educational     0.8462    0.7971    0.8209        69
        Neutral     0.8406    0.8169    0.8286        71
Overstimulating     0.8289    0.9000    0.8630        70

       accuracy                         0.8381       210
      macro avg     0.8386    0.8380    0.8375       210
   weighted avg     0.8385    0.8381    0.8375       210

=======================================================

Success! New 'nb_model.pkl' saved to C:\Users\Comsci and Crypto\Desktop\Sample\CF-2\backend\app\models

Microsoft Windows [Version 10.0.19045.6466]
(c) Microsoft Corporation. All rights reserved.

C:\Users\Comsci and Crypto\Desktop\Sample\CF-2\ml_training\scripts>py 7test_hybridfusion.py

=================================================================
CHILDFOCUS — HYBRID EVALUATION  (v5 — 210 videos, config-aligned)
=================================================================
Source       : test_clean.csv — NB NEVER saw these 210 videos
Fusion       : v5-confidence-gated-aligned
BASE_ALPHA   : 0.6 (NB conf >= 0.4)
LOW_ALPHA    : 0.15 (NB conf <  0.4)
H_OVERRIDE   : Score_H < 0.07 → cannot be Overstimulating
Thresholds   : Block >= 0.3 | Allow <= 0.12
Cookies      : ✓ found
Sample       : 70 per class = 210 total

[HYBRID] Loaded 210 videos from test_clean.csv
[HYBRID]   Educational: 69
[HYBRID]   Neutral: 71
[HYBRID]   Overstimulating: 70
[HYBRID]   Selected 69 Educational videos
[HYBRID]   Selected 70 Neutral videos
[HYBRID]   Selected 70 Overstimulating videos
[HYBRID] Resuming — 14 already done.

[HYBRID] Videos to process: 195 (14 already done)


[ 15/210] video_id=MG10BPnR53k | class=Overstimulating
        title: 'Who Is Eating Sugar Secretly? 🍬🤫 #Shorts #KidsStory #FunLearning'

[SAMPLER] ══════════════════════════════════════
[SAMPLER] Analyzing: MG10BPnR53k
[SAMPLER] Found cookies.txt, using preemptively to bypass bot-blocks.
[SAMPLER] Download attempt 1: 6.3s
[SAMPLER] ✓ 'Who Is Eating Sugar Secretly? 🍬🤫 #Shorts #KidsStory #FunLearning' (22.4s)
[SAMPLER] Short video (22s) — ad-skip disabled
[SAMPLER] Segments: [('S1', 0), ('S2', 1), ('S3', 2)] | seg_dur=20s | ad_skip=0s
[SAMPLER] Thumbnail: 0.1823
[SAMPLER] Analysis: 19.2s
[SAMPLER] ✓ Score: 0.2086 → Neutral
[SAMPLER] ✓ Total runtime: 25.8s
[SAMPLER] ══════════════════════════════════════

[403/210] video_id=L8A4XbM5sXA | class=Educational
        title: 'Dora Becomes a Doctor! 🩺 | FULL EPISODE "Doctor Dora" | Dora the '

[SAMPLER] ══════════════════════════════════════
[SAMPLER] Analyzing: L8A4XbM5sXA
[SAMPLER] Found cookies.txt, using preemptively to bypass bot-blocks.
[SAMPLER] Download attempt 1: 41.8s
[SAMPLER] ✓ 'Dora Becomes a Doctor! 🩺 | FULL EPISODE "Doctor Dora" | Dora the Explorer' (1421.8s)
[SAMPLER] Ad-skip: first 15s excluded from analysis
[SAMPLER] Segments: [('S1', 15), ('S2', 29), ('S3', 43)] | seg_dur=20s | ad_skip=15s
[SAMPLER] Thumbnail: 0.4
[SAMPLER] Analysis: 1.4s
[SAMPLER] ✓ Score: 0.1103 → Safe
[SAMPLER] ✓ Total runtime: 43.2s
[SAMPLER] ══════════════════════════════════════

[YTAPI] ✓ Scraped 28 hidden keywords for L8A4XbM5sXA
  [NB] Tags: 28 + 28 scraped = 28
[NB] 'Dora Becomes a Doctor! 🩺 | FULL EPISODE "Doct' → Educational | Score_NB=0.245 | P(over)=0.245
  ✗ [    Educational →         Neutral] NB=0.245(conf=0.60) H=0.110 α=0.6 Final=0.191 [FULL] (46.48s) 'Dora Becomes a Doctor! 🩺 | FULL EPI'
        Elapsed: 116.6min | ETA: ~0min remaining

[HYBRID] All 209 videos processed.

=================================================================
HYBRID EVALUATION RESULTS  (v5 — 210-video test_clean.csv)
=================================================================
Total attempted  : 209
Valid results    : 209
Skipped / Error  : 0

Fusion v5-confidence-gated-aligned
BASE_ALPHA=0.6 (conf>=0.4) | LOW_ALPHA=0.15 | H_OVERRIDE=0.07
Block >= 0.3 | Allow <= 0.12

3-CLASS CLASSIFICATION REPORT:
                 precision    recall  f1-score   support

    Educational     0.7692    0.1449    0.2439        69
        Neutral     0.4882    0.8857    0.6294        70
Overstimulating     0.8406    0.8286    0.8345        70

       accuracy                         0.6220       209
      macro avg     0.6993    0.6197    0.5693       209
   weighted avg     0.6990    0.6220    0.5708       209

3-Class Confusion Matrix:
                             Educa         Neutr         Overs
         Educational            10            53             6
             Neutral             3            62             5
     Overstimulating             0            12            58

── 3-Class Summary ──
Overall Accuracy   : 0.6220  (62.20%)
Macro Precision    : 0.6993
Macro Recall       : 0.6197
Macro F1           : 0.5693

── Binary Summary (Safe vs Overstimulating) ──
                 precision    recall  f1-score   support

           Safe     0.8406    0.8286    0.8345        70
Overstimulating     0.9143    0.9209    0.9176       139

       accuracy                         0.8900       209
      macro avg     0.8774    0.8747    0.8760       209
   weighted avg     0.8896    0.8900    0.8898       209

Binary Confusion Matrix:
                                  Safe   Overstimulating
                Safe               128                11
     Overstimulating                12                58
Binary Accuracy    : 0.8900  (89.00%)

── Child Safety Metric ──
Overstimulating Recall : 0.8286 (82.86%)
Missed detections      : 11 out of 70

── Path Breakdown ──
Full video analysis  : 206
Thumbnail-only       : 3
H_OVERRIDE triggered : 8  (Score_H < 0.07)
LOW_ALPHA used       : 23  (NB conf < 0.4)

[HYBRID] Results JSON → C:\Users\Comsci and Crypto\Desktop\Sample\CF-2\ml_training\scripts\..\outputs\hybrid_210_results.json
[HYBRID] Report TXT  → C:\Users\Comsci and Crypto\Desktop\Sample\CF-2\ml_training\scripts\..\outputs\hybrid_210_report.txt

=================================================================
THESIS NUMBERS — PASTE INTO CHAPTER 5 SECTION 5.3.3
=================================================================
Source        : test_clean.csv (209 valid / 209 attempted)
Fusion        : v5-confidence-gated-aligned
3-class acc   : 0.6220 (62.20%)
Macro F1      : 0.5693
Binary acc    : 0.8900 (89.00%)
Overst. recall: 0.8286

Per-class breakdown:
  Educational       : P=0.7692  R=0.1449  F1=0.2439  n=69
  Neutral           : P=0.4882  R=0.8857  F1=0.6294  n=70
  Overstimulating   : P=0.8406  R=0.8286  F1=0.8345  n=70

3-Class Confusion Matrix:
                             Educa         Neutr         Overs
         Educational            10            53             6
             Neutral             3            62             5
     Overstimulating             0            12            58

Binary Confusion Matrix:
                                  Safe   Overstimulating
                Safe               128                11
     Overstimulating                12                58
=================================================================

[HYBRID] Complete.

C:\Users\Comsci and Crypto\Desktop\Sample\CF-2\ml_training\scripts>py 8post_test_hyperparameter.py

======================================================================
CHILDFOCUS — FINAL HYBRID CONFIGURATION SEARCH
======================================================================
[FINAL] Loaded 209 valid video results

======================================================================
CONFIGURATION COMPARISON SUMMARY
======================================================================

Configuration                    Alpha   Block   Allow        F1   Accuracy
---------------------------------------------------------------------------
Original (thesis)                  0.4   0.750   0.350    0.1810     0.3254
Recalibrated basic                 0.4   0.185   0.110    0.4577     0.5311
NB-dominant (0.7 alpha)            0.7   0.300   0.150    0.5760     0.6124
NB-dominant (0.8 alpha)            0.8   0.300   0.150    0.5717     0.6124
NB-dominant (0.9 alpha)            0.9   0.300   0.150    0.5607     0.6029

======================================================================
GRID SEARCH — ALPHA × THRESHOLD COMBINATIONS
======================================================================

   Alpha    Block    Allow         F1   Accuracy
--------------------------------------------------
     0.5    0.280    0.180     0.6108     0.6124 ← BEST
     0.4    0.250    0.170     0.6107     0.6172
     0.6    0.280    0.170     0.6011     0.6172
     0.4    0.250    0.160     0.6010     0.6124
     0.6    0.280    0.180     0.6008     0.6124
     0.4    0.250    0.180     0.6006     0.6077
     0.6    0.300    0.180     0.5997     0.6029
     0.6    0.300    0.170     0.5992     0.6077
     0.5    0.280    0.170     0.5977     0.6029
     0.6    0.320    0.180     0.5967     0.5933

======================================================================
BEST CONFIGURATION — DETAILED REPORT
======================================================================

Alpha (NB weight)    : 0.5  (50% NB / 50% Heuristic)
Block threshold      : >= 0.28  (Overstimulating)
Allow threshold      : <= 0.18  (Educational)
Neutral range        : 0.18 < score < 0.28

Per-video results:
      video_id             true             pred     NB      H   Final
  -----------------------------------------------------------------
  ✗  gBUuQwN-0TM          Neutral      Educational  0.155  0.198  0.1769  '🐭 Learn to Draw a Cute Mouse S'
  ✗  GJuO4MzYrtI      Educational          Neutral  0.212  0.170  0.1910  'Thank You Teachers! | SciShow '
  ✗  hNFppd7L0Bg  Overstimulating          Neutral  0.489  0.068  0.2786  'OMG CAPCUT HAS SMOOTH VELOCITY'
  ✓  aMGQtiQUdkA  Overstimulating  Overstimulating  0.673  0.183  0.4279  'Baby Shark - Best Kids Dance V'
  ✓  kDdg2M1_EuE      Educational      Educational  0.182  0.111  0.1469  'The Alphabet Is So Much Fun | '
  ✓  _fsjQXoqFtM          Neutral          Neutral  0.271  0.179  0.2251  'Precious Plant GOES TO THE BIN'
  ✗  yc0GQ0tL0FE      Educational          Neutral  0.167  0.202  0.1847  'ABC Song for Babies &amp; Kids'
  ✓  mXMofxtDPUQ      Educational      Educational  0.230  0.101  0.1654  'Days of the Week Song | The Si'
  ✗  PVFNfmoH6pg          Neutral      Educational  0.117  0.158  0.1375  'How To Draw A Cute Husky With '
  ✗  DvpQHdQFJdE          Neutral      Educational  0.139  0.153  0.1459  'Study vlog 📚 ˙✧˖° #study #stud'
  ✓  VVgu67WS0DI          Neutral          Neutral  0.182  0.203  0.1928  'The Seven Little Goats and The'
  ✓  RM485oUuOhg          Neutral          Neutral  0.244  0.154  0.1991  'Switzerland: The Land of Pure '
  ✓  J5hKPShTi3E          Neutral          Neutral  0.301  0.172  0.2366  '4 Ways To Get Organized'
  ✗  UiFjZcOrMO0      Educational  Overstimulating  0.537  0.212  0.3744  'Kaboochi | Dance Song For Kids'
  ✓  MG10BPnR53k  Overstimulating  Overstimulating  0.606  0.209  0.4072  'Who Is Eating Sugar Secretly? '
  ✗  R0vgSne0jE8  Overstimulating          Neutral  0.221  0.190  0.2054  'Shapes are EVERYWHERE #shorts'
  ✓  PyOmuhdibVM          Neutral          Neutral  0.220  0.153  0.1862  'Nightly Routine #cleaning #apa'
  ✗  lWhqORImND0      Educational          Neutral  0.364  0.160  0.2621  'Old MacDonald Had A Farm + Mor'
  ✗  0W86K1jBJFI          Neutral      Educational  0.187  0.157  0.1719  'Little Red Riding Hood - Anima'
  ✓  2bUTxSPyCzo  Overstimulating  Overstimulating  0.682  0.215  0.4483  'Hide and Seek in One Color wit'
  ✗  UpDPJICdhhg      Educational          Neutral  0.418  0.001  0.2095  '6.6 LogicLike - Educational ga'
  ✓  q2PglyGrHqc  Overstimulating  Overstimulating  0.872  0.088  0.4799  'Baby Shark doo doo doo #shorts'
  ✓  0LnhNC_wHY8  Overstimulating  Overstimulating  0.436  0.200  0.3180  'Rain, Rain, Go Away Nursery Rh'
  ✓  _OcW8ZHGZr8      Educational      Educational  0.041  0.142  0.0914  'Back To School Reading for Kid'
  ✗  G8nuZu0VDp4      Educational          Neutral  0.243  0.176  0.2093  '5 Minute Stories Book Read-Alo'
  ✗  LDD6hX0lpj8          Neutral      Educational  0.152  0.137  0.1447  'aesthetic morning routine 🌸 #a'
  ✓  viurF6fz9Xg  Overstimulating  Overstimulating  0.443  0.194  0.3180  '$1 VS $1,000,000 99 NIGHTS IN '
  ✓  49df3lEIYyg  Overstimulating  Overstimulating  0.468  0.191  0.3294  'Baby Got a Boo Boo | Nursery R'
  ✓  6QdwB7ZCyMA      Educational      Educational  0.021  0.135  0.0779  '@Numberblocks- Meet Number Nin'
  ✗  daL2STCfFHw  Overstimulating          Neutral  0.343  0.215  0.2793  'Building Our Family Island in '
  ✓  xoamCzuQjy0      Educational      Educational  0.220  0.094  0.1565  'Colors Vocabulary Chant for Ki'
  ✗  c_NGdBIlu0M  Overstimulating          Neutral  0.260  0.144  0.2021  'aayu and pihu show 👦😍👩#shortvi'
  ✓  dmVNCtbcTdU  Overstimulating  Overstimulating  0.595  0.186  0.3904  'Hot Wheels Monster Trucks 1:6 '
  ✗  l_SSeAMwCDw          Neutral      Educational  0.122  0.151  0.1365  'Very Easy! Fish Drawing For Ki'
  ✓  tisxrc_6zbE  Overstimulating  Overstimulating  0.680  0.234  0.4571  'Chris collects toys for his ga'
  ✓  qJIYkbAgR8c  Overstimulating  Overstimulating  0.465  0.183  0.3239  'Yes Yes Vegetables | 🥕 The Hun'
  ✗  bt3ttH11QT8  Overstimulating          Neutral  0.338  0.159  0.2483  'Smashing a 34,000 Brick Statue'
  ✓  XOnHtStmbCI          Neutral          Neutral  0.290  0.200  0.2451  'Mickey and Donald Have a Farm '
  ✓  6M3IrkyNNT8      Educational      Educational  0.022  0.167  0.0944  '@Numberblocks- Meet Number Fiv'
  ✗  LSfqhUKf0X0  Overstimulating          Neutral  0.321  0.207  0.2639  'Villains are taking over! We n'
  ✓  FXjC69MmcPY      Educational      Educational  0.183  0.143  0.1633  '🌈🎵 Learn, Sing, and Explore! P'
  ✓  gB1dNEkogYk      Educational      Educational  0.245  0.056  0.1505  'Certified BOP'
  ✓  3JGOK5q5IMA          Neutral          Neutral  0.298  0.217  0.2578  '3-Ingredient Nutella Cookies! '
  ✓  0j_AbvBOWZw  Overstimulating  Overstimulating  0.530  0.116  0.3233  'More Baby Seals for You (Part3'
  ✗  s0bS-SBAgJI          Neutral      Educational  0.199  0.100  0.1496  'The Water Cycle- How rain is f'
  ✗  hkz51Td9k5A          Neutral      Educational  0.158  0.178  0.1683  "Read Aloud Kids Book: It's A F"
  ✗  sOxloXyOAKA          Neutral      Educational  0.247  0.067  0.1569  'Nature View Cristal beach Beut'
  ✓  gDInVjBUGC0      Educational      Educational  0.132  0.057  0.0941  'ELS: Phase 3 pronunciation'
  ✓  AGYPvz9Dov8          Neutral          Neutral  0.220  0.174  0.1968  'HIPPO VIDEOS FOR KIDS - Facts '
  ✗  r2bW4ySQr-4          Neutral      Educational  0.228  0.065  0.1465  'Creative Art For Kids Chair 🪑🌈'
  ✗  xw4BPniE2m4      Educational          Neutral  0.339  0.163  0.2512  'This Ruthless Spider Is the Ma'
  ✓  fhPaQwEoGGM  Overstimulating  Overstimulating  0.782  0.192  0.4869  'Kids Trapped in Minecraft!'
  ✗  LpUzg3Dc-m0      Educational          Neutral  0.309  0.193  0.2510  'Never Take Seashells From Beac'
  ✗  rbVkRDOjMa8  Overstimulating          Neutral  0.344  0.189  0.2667  'Sunny Bunnies | Shiny Bright B'
  ✗  qaRYVwND3SQ          Neutral      Educational  0.166  0.169  0.1673  'How to Blow Your Nose - Life S'
  ✓  89QRrnnYPNw          Neutral          Neutral  0.282  0.205  0.2438  'How Do Plants Grow? | Knowsy N'
  ✗  fKigjAnJrjg      Educational          Neutral  0.255  0.191  0.2230  'The Magic Magnet | Funny Stori'
  ✗  xfLYZeLV3go      Educational          Neutral  0.244  0.183  0.2133  'The Egg&#39;s Adventure | Educ'
  ✗  nk6QEJkNWHc          Neutral      Educational  0.163  0.150  0.1567  'A day in my cottage life'
  ✗  GKLRkTA64N0      Educational          Neutral  0.284  0.094  0.1891  'Numbers Silly Song'
  ✓  mepSmUC6HWM  Overstimulating  Overstimulating  0.682  0.222  0.4518  'Monsters Play Trash Basketball'
  ✓  zdQfvoKNLhE  Overstimulating  Overstimulating  0.482  0.230  0.3561  'Oh! Hero Piggy Saved the Littl'
  ✗  rtAwdYTjNhY      Educational          Neutral  0.293  0.131  0.2116  "Quick Quiz! Who's Larger? Male"
  ✓  C4l7rfjX49I          Neutral          Neutral  0.414  0.098  0.2561  'Who won? #shorts #family'
  ✗  k14YDC1Kn-o          Neutral      Educational  0.174  0.177  0.1758  'The Lion and The Mouse | Fairy'
  ✓  IzkmfRTktWc      Educational      Educational  0.219  0.087  0.1529  'Building Squares from Odd Numb'
  ✓  I_4Xw57Lt-k  Overstimulating  Overstimulating  0.783  0.184  0.4835  'Kids Cars Adventures Compilati'
  ✗  OtalQpNt2UQ      Educational          Neutral  0.277  0.183  0.2298  'Dora & Boots Go On a Puppy Adv'
  ✓  aBAOUEVdNJA          Neutral          Neutral  0.315  0.151  0.2330  'those school days... 😭🌷 #short'
  ✓  nkqet4ca25A          Neutral          Neutral  0.330  0.154  0.2419  'REALITY vs SLOW MO! 🎥'
  ✗  7SWvlUd2at8          Neutral      Educational  0.107  0.087  0.0968  '10 Easy Animal Drawings for Ki'
  ✓  VdzzE20zQC8      Educational      Educational  0.205  0.142  0.1734  'The Shapes Song | Nursery Rhym'
  ✓  q795N2khTXk  Overstimulating  Overstimulating  0.520  0.132  0.3260  'Would You Let Your Kids Play T'
  ✗  Rw3NxD-p8_U  Overstimulating          Neutral  0.349  0.179  0.2640  'Possible 1x1x1x1?? recent "hac'
  ✗  yTXtTX3t8fM      Educational  Overstimulating  0.418  0.147  0.2823  '9.1 LogicLike - Educational ga'
  ✓  qRDSgx5PF9g          Neutral          Neutral  0.315  0.175  0.2449  'Pictures that relieve anxiety!'
  ✗  R_CD6ellkyk          Neutral      Educational  0.259  0.056  0.1579  'Relaxing Video of a Sunrise Be'
  ✗  QsoRCSnhsQo      Educational          Neutral  0.228  0.153  0.1906  'Return Little Crocodile Home |'
  ✓  MvXOhpSuTx4  Overstimulating  Overstimulating  0.628  0.196  0.4121  'Shoo Be Doo Birthday🥳🎈# birthda'
  ✓  9w9GlT4Zyl8          Neutral          Neutral  0.375  0.180  0.2774  'Wall&#39;s 3 in 1 Ice Cream 🍦 '
  ✓  AS2gsXyn4ts  Overstimulating  Overstimulating  0.450  0.174  0.3125  "A Cute Zombie's Message Just f"
  ✓  BfodhLizt_A  Overstimulating  Overstimulating  0.712  0.189  0.4506  'Unboxing Disney Cars Tomica Ra'
  ✓  0JDCViWGn-0          Neutral          Neutral  0.293  0.183  0.2383  'Human Body Systems Overview (U'
  ✓  -PxShNDmIHw      Educational      Educational  0.043  0.083  0.0626  "Let's Spell Words ending in 'a"
  ✓  _f1QbJm1irY          Neutral          Neutral  0.196  0.167  0.1816  'The Smartest Giant in Town - A'
  ✓  Ae4MadKPJC0          Neutral          Neutral  0.247  0.127  0.1868  'Human Body 101 | National Geog'
  ✗  36n93jvjkDs      Educational          Neutral  0.236  0.193  0.2142  'Days of The Week Song For Kids'
  ✓  4DoO_Td6k8M  Overstimulating  Overstimulating  0.700  0.225  0.4625  'Behind Story of Pinkfong Baby '
  ✓  _Dwbf5WjjJA  Overstimulating  Overstimulating  0.396  0.179  0.2873  'IMPOSTERS are PRANKING US! FUL'
  ✓  -G49u2y-0U4          Neutral          Neutral  0.357  0.084  0.2205  'Mom and Dad use girls as mops '
  ✓  lmRmk2HrqYA          Neutral          Neutral  0.386  0.001  0.1937  'Animal Showdown 🐻🐊🦅 | 25 Minut'
  ✓  JhH-_QXzQ3o  Overstimulating  Overstimulating  0.467  0.118  0.2921  'Baby Seal: Not Home Alone (Ful'
  ✓  v3BCkHGnKT4  Overstimulating  Overstimulating  0.592  0.187  0.3892  'Andrea Becomes Superheroes to '
  ✓  2K4o11uxhs0      Educational      Educational  0.087  0.183  0.1350  'ABC Phonics Song + More Nurser'
  ✗  bWAIgN_DTxo      Educational          Neutral  0.283  0.091  0.1867  'Everything You Need to Know Ab'
  ✓  MmgFk1oL9a8          Neutral          Neutral  0.277  0.084  0.1805  '🥹how beautiful is this � ##viral'
  ✓  K0GujQE6Jqc      Educational      Educational  0.142  0.167  0.1547  'ABC Phonics Song - A for Apple'
  ✓  CUBciV4iPzs          Neutral          Neutral  0.374  0.175  0.2743  'Baking With Blippi | Food Vide'
  ✓  U9Q7Y3t4m3g      Educational      Educational  0.214  0.121  0.1673  'Good Morning Song For Children'
  ✗  0GAhHaK4y58      Educational          Neutral  0.304  0.170  0.2370  'Blippi Meets TALKING PLANETS! '
  ✗  mDzIA3VUFKI  Overstimulating          Neutral  0.388  0.151  0.2693  'Lunar New Year Dance 💃👯🏮 Talki'
  ✓  XSun4GUyhuY      Educational      Educational  0.105  0.222  0.1634  'Preschool Learning activities '
  ✓  QK_4HL1vK0o  Overstimulating  Overstimulating  0.655  0.225  0.4400  'Escaping 100 Layers of CANDY C'
  ✗  3CfvBCAPZyE      Educational          Neutral  0.302  0.182  0.2424  'Ms. Rachel Counts to 10 with B'
  ✗  tSEhQO0zlGQ          Neutral  Overstimulating  0.564  0.184  0.3743  'Thomas the Rescue Engine | Car'
  ✓  Mbc29yUtIxs          Neutral          Neutral  0.302  0.206  0.2538  'Microwave Meals: Lava Cake!'
  ✓  1szs8RMlhN8          Neutral          Neutral  0.331  0.174  0.2527  '1 carrot with 1 egg! your kids'
  ✗  y8kneKeNyCI          Neutral  Overstimulating  0.508  0.189  0.3483  'Belly Button Song Dance! Learn'
  ✓  NQOD8oBfZjs          Neutral          Neutral  0.218  0.168  0.1930  'A Day In The Life Of Serengeti'
  ✓  g2jdZ46nK-M      Educational      Educational  0.193  0.156  0.1742  'Shapes Song For Kids-Circle, T'
  ✓  fTYhA_j6cUw  Overstimulating  Overstimulating  0.783  0.147  0.4649  'Water Safety Song | Run Away f'
  ✗  X0SpQyIHVXQ      Educational          Neutral  0.312  0.171  0.2415  'What is in the fridge? 🐧'
  ✓  zTmjQz198f0  Overstimulating  Overstimulating  0.567  0.155  0.3611  'Fish 123 | Ten Little Fish | L'
  ✓  YSA2f19ABPk  Overstimulating  Overstimulating  0.678  0.194  0.4359  'Chris and friends in Magic Esc'
  ✓  aJxxbI9HIRU  Overstimulating  Overstimulating  0.676  0.208  0.4424  'Baby Shark - Sing and Dance - '
  ✓  bjRpNYq8Ts0          Neutral          Neutral  0.246  0.156  0.2012  'Exploring the Adorable World o'
  ✓  THYU9WgZD_Y  Overstimulating  Overstimulating  0.357  0.210  0.2834  'This is going to be AWESOME! #'
  ✓  hV5eWushhbQ  Overstimulating  Overstimulating  0.428  0.133  0.2807  '3 Skeletons Were Riding On A B'
  ✗  ao2IP7f23-8          Neutral      Educational  0.125  0.135  0.1299  'How To Draw A Hibiscus Flower '
  ✗  vInTXlOIRzQ          Neutral      Educational  0.130  0.101  0.1153  'How To Draw Cinnamoroll From H'
  ✓  RjpRL6tv3Cw          Neutral          Neutral  0.388  0.111  0.2499  'Bluey and Bingo Pretend to be '
  ✓  KGD1TJ_XjbI          Neutral          Neutral  0.321  0.203  0.2621  'Korean Cheesy Potato Pancake ('
  ✗  WhhPGrWxWsI      Educational          Neutral  0.216  0.201  0.2082  'Harbour Learning | Sea Animal '
  ✓  4HGwX5b44Yc  Overstimulating  Overstimulating  0.693  0.212  0.4526  'Superheroes Teamwork Adventure'
  ✗  SNF8b7KKJ2I          Neutral      Educational  0.181  0.127  0.1538  'Ecosystems for Kids | Plants, '
  ✓  oY-H2WGThc8          Neutral          Neutral  0.313  0.170  0.2417  'Clean Up Song for Children - b'
  ✓  qCs0pm_D_hM  Overstimulating  Overstimulating  0.588  0.167  0.3775  'Colors for Children | Toy Trai'
  ✓  IPr_Ay-7aT0      Educational      Educational  0.201  0.101  0.1510  'The Phonics Song + More Presch'
  ✗  7aKsPwH1iss          Neutral      Educational  0.225  0.073  0.1492  'beautiful sunset video #shorts'
  ✓  1nJ5VbyD6tY      Educational      Educational  0.180  0.153  0.1667  '2D Shapes Song'
  ✗  qqwRkhwrJek          Neutral  Overstimulating  0.381  0.203  0.2920  'Jingle Bells,  Christmas Song,'
  ✓  D5XMzXizUKA  Overstimulating  Overstimulating  0.856  0.171  0.5135  '60 Minutes Ultimate Cooking To'
  ✓  UvVKtqlxJq8  Overstimulating  Overstimulating  0.440  0.187  0.3138  'Try NOT To Laugh: Silly Jokes '
  ✓  xgJjg4HSo4A          Neutral          Neutral  0.256  0.178  0.2169  'Organize – Teach Kids to Stay '
  ✓  PL7vHFGRViA  Overstimulating  Overstimulating  0.775  0.188  0.4817  'Satisfying with Unboxing & Rev'
  ✗  2jMyZ0OZan8          Neutral      Educational  0.113  0.164  0.1382  'How To Draw The Cutest Fish Us'
  ✓  zZz91aicTuU  Overstimulating  Overstimulating  0.497  0.179  0.3383  'Power Rangers Ninja Kidz Battl'
  ✓  WkbfeCays4w  Overstimulating  Overstimulating  0.765  0.192  0.4781  'Baby Shark Sing and Dance Song'
  ✗  0n_J2z-ILXo      Educational          Neutral  0.385  0.143  0.2643  'Humpty Dumpty | + More Kids So'
  ✓  geyzFEhOu_s  Overstimulating  Overstimulating  0.557  0.207  0.3816  'Rich Princess Treats the Princ'
  ✗  NkY57mizvMI      Educational          Neutral  0.224  0.167  0.1955  '🐯 Meet TINKER! 🐯 | Your Favori'
  ✗  b2sjCyAsFbM      Educational          Neutral  0.270  0.163  0.2167  'Peter Rabbit - Camping by the '
  ✓  vXkY5wIWu18  Overstimulating  Overstimulating  0.544  0.177  0.3609  'Kaden Eva & Emma Jump in a Hol'
  ✓  9M4JWcKEdL0      Educational      Educational  0.262  0.092  0.1770  'Three Space Weather Missions L'
  ✓  cixocSycC3A  Overstimulating  Overstimulating  0.475  0.153  0.3140  'So Satisfying 😳'
  ✓  eg2mr1IticM  Overstimulating  Overstimulating  0.537  0.158  0.3473  'EASTER EGG ASMR! 🐣💚🩷🩵   #asmr #a'
  ✓  e2JMt_YJveM          Neutral          Neutral  0.231  0.156  0.1932  'What are clouds? ☁☁ How are th'
  ✓  1y8MZvPhCKE      Educational      Educational  0.042  0.123  0.0824  'The Best of Alphablock O  | Le'
  ✓  Kx241A5WsDI  Overstimulating  Overstimulating  0.584  0.189  0.3869  'Blippi & Yellowcard GO GO GO o'
  ✗  eclBbnyOsYA          Neutral  Overstimulating  0.469  0.165  0.3167  'Learn Letters! - ABC Undersea '
  ✓  ki5wRkJWPP8  Overstimulating  Overstimulating  0.697  0.201  0.4486  'Vlad plays with magic masks'
  ✗  CGSJCE0Rqnw      Educational          Neutral  0.293  0.178  0.2352  'Is laughter ACTUALLY the best '
  ✓  pr5U8x-kCMM  Overstimulating  Overstimulating  0.421  0.166  0.2934  'LARVA - RAINING | Larva 2017 |'
  ✗  ozsgl_sLnHQ      Educational          Neutral  0.168  0.224  0.1961  'Learn About Emotions and Feeli'
  ✓  qoXM4iykEz8          Neutral          Neutral  0.208  0.178  0.1933  "How to Get a Good Night's Slee"
  ✗  gbsgbcd_-2U          Neutral  Overstimulating  0.417  0.169  0.2932  'PAW Patrol Rescue Wheels Adven'
  ✓  -gajDGMq7Lg  Overstimulating  Overstimulating  0.405  0.186  0.2954  'When a girl gets you angry 😂 #'
  ✗  Oyu80bhbrNk      Educational          Neutral  0.249  0.142  0.1955  'Screen Time That Earns Its Pla'
  ✓  KyBYuEgvFl0  Overstimulating  Overstimulating  0.764  0.167  0.4657  'Baby Car | Car Songs | Pinkfon'
  ✓  mJaxCjNJDww      Educational      Educational  0.100  0.161  0.1304  'Learning Videos for Toddlers |'
  ✓  KsaGjQDxjJQ  Overstimulating  Overstimulating  0.734  0.215  0.4748  '30 Minutes Satisfying with Unb'
  ✓  ws_D5nXOAJg          Neutral          Neutral  0.247  0.128  0.1875  'The Stunning Life Cycle Of A L'
  ✓  Ov-KDfyUR0A  Overstimulating  Overstimulating  0.608  0.235  0.4217  'Vlad and Niki build a construc'
  ✗  CYEyoUUfA8A      Educational          Neutral  0.248  0.128  0.1880  'Routines That Build More Than '
  ✓  FjtyU_KNits      Educational      Educational  0.103  0.160  0.1310  'Learn the Alphabet Sounds with'
  ✗  _C-cxckDJX0          Neutral      Educational  0.159  0.140  0.1497  'morning routine #shorts'
  ✓  r1Hbzt_tjGg          Neutral          Neutral  0.283  0.179  0.2310  'Cute Rainbow Painting Hack for'
  ✓  fdm1_QInzfM          Neutral          Neutral  0.179  0.188  0.1832  'How To Draw An Emoji Folding S'
  ✗  ZY-NF22bKXY          Neutral      Educational  0.131  0.182  0.1564  'Aesthetic Quiet Morning Study '
  ✓  Lvkf2bwch7w          Neutral          Neutral  0.247  0.137  0.1920  'My morning routine #shorts'
  ✓  gV0w_wrU750  Overstimulating  Overstimulating  0.583  0.175  0.3793  '&quot;Baby Shark&quot; - The P'
  ✓  10OJhIc_-vw  Overstimulating  Overstimulating  0.459  0.184  0.3219  'Minecraft Speedrunner VS 6 Hun'
  ✓  23C5twKp72Q      Educational      Educational  0.198  0.106  0.1522  "Meet the Shapes | 'Rectangle' "
  ✗  Me9Ex4QZnbU      Educational  Overstimulating  0.595  0.192  0.3936  'Wheels On The Bus story with A'
  ✗  EoRXcJRtKq4          Neutral      Educational  0.149  0.173  0.1607  'How to draw a cute girl easy |'
  ✓  zHT2nY-Gq1I  Overstimulating  Overstimulating  0.550  0.212  0.3810  'Kids Learn About All Their Fee'
  ✗  r4MfjpBQCvQ      Educational  Overstimulating  0.407  0.153  0.2801  'The Shark is Coming+More | Yum'
  ✓  l5RrO6cYy_A          Neutral          Neutral  0.205  0.185  0.1952  'day in my life as a college st'
  ✓  zugFjQ8tN3E  Overstimulating  Overstimulating  0.440  0.187  0.3130  'Eva Emma & Kaden Playground Ge'
  ✓  yeVTvh9HU8s  Overstimulating  Overstimulating  0.447  0.142  0.2941  'Blippi Meets Dr. Ted the Puppy'
  ✗  3ZHYQ6f1BhU      Educational          Neutral  0.234  0.214  0.2240  'Cavities - The Dr. Binocs Show'
  ✗  quUlfkOyH0g          Neutral      Educational  0.232  0.103  0.1676  'Duck Drawing | Drawing for kid'
  ✗  pfRuLS-Vnjs      Educational          Neutral  0.256  0.113  0.1844  'The Shapes Song'
  ✗  s_hAnPdkBTI          Neutral      Educational  0.182  0.132  0.1573  'How to wake up in the morning '
  ✓  MfIwF6QoRm4  Overstimulating  Overstimulating  0.835  0.189  0.5117  'Police Officer Visits Doctors '
  ✓  Nx9rbyz3iDc          Neutral          Neutral  0.342  0.185  0.2636  'What really happens when we co'
  ✓  0VLxWIHRD4E      Educational      Educational  0.217  0.075  0.1463  "Let's Count to 20 Song For Kid"
  ✗  DS_PQYIGrTc          Neutral      Educational  0.183  0.085  0.1340  'How To Draw The Pusheen Cat'
  ✗  hCaHFv2YWn8  Overstimulating          Neutral  0.324  0.082  0.2029  'Lincoln Loud&#39;s Reaction! 😬'
  ✓  7wNyw7DlY0A          Neutral          Neutral  0.231  0.144  0.1880  'How to study when you are tire'
  ✓  jiEv6VTDt5c      Educational      Educational  0.093  0.132  0.1125  'Learn to Read - 3-Letter Word '
  ✗  r8iGnwD8i9I      Educational  Overstimulating  0.470  0.173  0.3216  'ABC Song and Many More Nursery'
  ✓  jOGuabdHYaE  Overstimulating  Overstimulating  0.823  0.153  0.4881  'Baby Shark Dance 2 | Sing and '
  ✗  g6CVApMxmqg      Educational          Neutral  0.237  0.190  0.2138  'Phonics, Counting, Colors + Mo'
  ✓  d2S87jXhlV0  Overstimulating  Overstimulating  0.737  0.186  0.4613  'Original Baby Shark | Go #Baby'
  ✓  v7cuoaoUTKQ  Overstimulating  Overstimulating  0.857  0.216  0.5365  'Baby Shark Dance | Pinkfong Si'
  ✗  KoDg_L6B2y4      Educational          Neutral  0.242  0.214  0.2283  'Let Them Cook: Collard Greens '
  ✓  Be4j6lJpFTQ      Educational      Educational  0.105  0.177  0.1409  'Counting to 20 Song For Kids |'
  ✗  kopx0X8XTMM      Educational          Neutral  0.271  0.146  0.2088  'Stand like a flamingo with Ms.'
  ✓  9h8RmsptqaI  Overstimulating  Overstimulating  0.412  0.183  0.2977  'New Chicken Dance #shorts  by '
  ✗  rifuEZFieaI      Educational          Neutral  0.189  0.212  0.2009  'SUSTAINABLE DEVELOPMENT |#clim'
  ✓  RxCwVQGlDis  Overstimulating  Overstimulating  0.781  0.103  0.4423  'Baby Shark Dance | DJ Raphi So'
  ✓  6VQip5MI5d8  Overstimulating  Overstimulating  0.808  0.173  0.4907  'Johny Johny Yes Papa + Wheels '
  ✓  LMuBxwVy3HM      Educational      Educational  0.221  0.132  0.1766  'Peppa Pig Full Episodes | Pott'
  ✗  qbIgrxIfHxI          Neutral      Educational  0.140  0.166  0.1530  'Beautiful Nature with Rural Li'
  ✗  e87xaEjYMtg      Educational          Neutral  0.261  0.129  0.1948  'Peanut Butter And Jelly �  🍇 #s'
  ✓  YI0bd8Tw8rE      Educational      Educational  0.026  0.098  0.0624  '@Numberblocks- Meet Number Sev'
  ✗  HrDl_1Ov8gc      Educational  Overstimulating  0.427  0.199  0.3133  'Learn Colors with Wonderville '
  ✓  L8A4XbM5sXA      Educational      Educational  0.245  0.110  0.1777  'Dora Becomes a Doctor! 🩺  | FUL'

Classification Report:
                 precision    recall  f1-score   support

    Educational     0.5263    0.4348    0.4762        69
        Neutral     0.4691    0.5429    0.5033        70
Overstimulating     0.8451    0.8571    0.8511        70

       accuracy                         0.6124       209
      macro avg     0.6135    0.6116    0.6102       209
   weighted avg     0.6139    0.6124    0.6108       209

Confusion Matrix:
                           Educa       Neutr       Overs
         Educational          30          33           6
             Neutral          27          38           5
     Overstimulating           0          10          60

==================================================
FINAL METRICS (Best Config)
==================================================
Accuracy           : 0.6124  (61.24%)
Weighted Precision : 0.6139
Weighted Recall    : 0.6124
Weighted F1-Score  : 0.6108

Per-class:
  Educational       : P=0.5263  R=0.4348  F1=0.4762  n=69
  Neutral           : P=0.4691  R=0.5429  F1=0.5033  n=70
  Overstimulating   : P=0.8451  R=0.8571  F1=0.8511  n=70

[FINAL] Report saved → C:\Users\Comsci and Crypto\Desktop\Sample\CF-2\ml_training\scripts\..\outputs\post-hyperparameter_report.txt

======================================================================
PASTE THESE INTO YOUR THESIS — CHAPTER 5
======================================================================
Fusion config  : alpha=0.5 (NB), beta=0.5 (Heuristic)
Thresholds     : Block >= 0.28, Allow <= 0.18
Accuracy       : 0.6124 (61.24%)
Precision      : 0.6139
Recall         : 0.6124
F1-Score       : 0.6108

Per-class breakdown:
  Educational       : P=0.5263  R=0.4348  F1=0.4762
  Neutral           : P=0.4691  R=0.5429  F1=0.5033
  Overstimulating   : P=0.8451  R=0.8571  F1=0.8511

Confusion Matrix:
                           Educa       Neutr       Overs
         Educational          30          33           6
             Neutral          27          38           5
     Overstimulating           0          10          60
======================================================================

================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================

===============================================================FULL 1-8 PIPELINE ===============================================================================
Microsoft Windows [Version 10.0.19045.6466]
(c) Microsoft Corporation. All rights reserved.

C:\Users\Comsci and Crypto\Desktop\Sample\CF-2\ml_training\scripts>py 1preprocess.py
[PREPROCESS] Loaded 700 rows from data/raw/final_700_enriched.csv
[PREPROCESS] ✓ 700 rows saved → data/processed/metadata_clean.csv
[PREPROCESS] Label distribution:
[PREPROCESS]   Overstimulating: 235 (33.6%)
[PREPROCESS]   Educational: 230 (32.9%)
[PREPROCESS]   Neutral: 235 (33.6%)

[PREPROCESS] Expected split (train_nb.py will confirm):
[PREPROCESS]   Train (70%): 490  |  Test (30%): 210
[PREPROCESS] Next step: python train_nb.py

C:\Users\Comsci and Crypto\Desktop\Sample\CF-2\ml_training\scripts>py 2baseline.py
====================================================================
  BASELINE RESULTS — All Four Metrics  (test_210.csv, n=210)
====================================================================
  Baseline                    Accuracy   Precision   Recall       F1   Over_Rec
  --------------------------------------------------------------------
  B1  Random chance             35.71%      35.67%    35.73%    35.66%      40.00%
  B2  Majority ('Overstimulating')    33.33%      11.11%    33.33%    16.67%     100.00%
  B3  Keyword heuristic         49.52%      60.46%    49.32%    47.56%      27.14%
====================================================================

  NOTE — Macro averages weight all 3 classes equally.
  Over_Rec = Overstimulating recall (child safety metric).

====================================================================
  B3 KEYWORD HEURISTIC — Per-Class Detail
====================================================================
                 precision    recall  f1-score   support

    Educational       0.61      0.39      0.48        69
        Neutral       0.41      0.82      0.54        71
Overstimulating       0.79      0.27      0.40        70

       accuracy                           0.50       210
      macro avg       0.60      0.49      0.48       210
   weighted avg       0.60      0.50      0.48       210

  Confusion Matrix  (rows=Actual, cols=Predicted):
                           Educa   Neutr   Overs
             Educational      27      41       1
                 Neutral       9      58       4
         Overstimulating       8      43      19

====================================================================
  YOUR MODEL MUST BEAT (all four metrics over baseline):
  Accuracy  > 49.52%    (B3 keyword heuristic)
  Precision > 60.46%    (macro)
  Recall    > 49.32%    (macro)
  F1        > 47.56%    (macro)
  Over_Rec  > 27.14%    (Overstimulating recall)
====================================================================

-> Run: py 3modelselection.py

C:\Users\Comsci and Crypto\Desktop\Sample\CF-2\ml_training\scripts>py 3model_selection.py
Training samples loaded: 490  <- must be 490
Label dist: {'Educational': 161, 'Neutral': 164, 'Overstimulating': 165}

==================================================================================
  Model                     Acc%   Prec%    Rec%      F1    ±Acc    Gap%
==================================================================================
  Complement NB           79.59%   79.74%   79.55%  0.7937  +-3.35%   15.9%
  Multinomial NB          79.18%   79.40%   79.14%  0.7895  +-3.45%   16.0%
  Logistic Regression     78.78%   79.40%   78.76%  0.7883  +-3.79%   19.1%
  Linear SVM              78.37%   78.80%   78.34%  0.7833  +-3.12%   21.6%
  Random Forest           73.88%   74.83%   73.90%  0.7405  +-6.66%   26.1%
==================================================================================

======================================================================
  COMPLEMENT NB — Per-Fold Detail (all four metrics)
======================================================================
  Fold       Accuracy   Precision   Recall       F1
  ------------------------------------------------
  Fold 1        84.69%      84.77%    84.63%  0.8466
  Fold 2        75.51%      75.56%    75.51%  0.7543
  Fold 3        81.63%      82.11%    81.60%  0.8162
  Fold 4        76.53%      76.74%    76.39%  0.7594
  Fold 5        79.59%      79.51%    79.64%  0.7919
  ------------------------------------------------
  Mean         79.59%      79.74%    79.55%  0.7937
  ±Std     +-   3.35%  +-   3.39%  +- 3.36%  +-0.0347

======================================================================
  MODEL SELECTION DECISION GUIDE
======================================================================
  Highest CV Accuracy   : Complement NB
  Highest Macro F1      : Complement NB
  Highest Macro Recall  : Complement NB
  Smallest Overfit Gap  : Complement NB

  Selected: Complement NB
  Reason  : Best or competitive on all four metrics with smallest
            overfit gap. Designed for complement class estimation
            (Rennie et al. 2003) — ideal for imbalanced 3-class text.

-> Test set (test_210.csv) still SEALED.
-> Next: py 4tunealpha.py

C:\Users\Comsci and Crypto\Desktop\Sample\CF-2\ml_training\scripts>py 4a_tune_alpha.py
Training samples loaded: 490  <- must be 490

================================================================================
     Alpha      Acc%     ±Acc    Prec%     Rec%        F1
================================================================================
  alpha=0.05      78.16%  +-2.10%   78.58%   78.11%  0.7802
  alpha=0.1       78.16%  +-1.66%   78.56%   78.11%  0.7801
  alpha=0.15      78.98%  +-1.38%   79.46%   78.93%  0.7884
  alpha=0.2       79.39%  +-1.76%   79.95%   79.34%  0.7926
  alpha=0.3       78.98%  +-2.10%   79.48%   78.94%  0.7884
  alpha=0.5       79.80%  +-2.77%   80.07%   79.76%  0.7966
  alpha=1.0       79.59%  +-3.35%   79.74%   79.55%  0.7937
  alpha=1.5       80.00%  +-3.14%   80.17%   79.96%  0.7975  <- BEST
  alpha=2.0       79.39%  +-2.77%   79.56%   79.35%  0.7916
================================================================================

======================================================================
  BEST ALPHA = 1.5 — Per-Fold Detail
======================================================================
  Fold       Accuracy   Precision   Recall       F1
  ------------------------------------------------
  Fold 1        84.69%      84.77%    84.63%  0.8466
  Fold 2        77.55%      77.63%    77.56%  0.7751
  Fold 3        82.65%      83.27%    82.61%  0.8261
  Fold 4        76.53%      76.74%    76.39%  0.7594
  Fold 5        78.57%      78.43%    78.63%  0.7804
  ------------------------------------------------
  Mean         80.00%      80.17%    79.96%  0.7975
  ±Std     +-   3.14%  +-   3.22%  +- 3.13%  +-0.0331

======================================================================
  STEP 4 COMPLETE
======================================================================
  Best alpha : 1.5
  CV Acc     : 80.00%
  CV F1      : 0.7975

-> Open 4b_tune_tfidf.py and confirm: BEST_ALPHA = 1.5
-> Then run: py 4b_tune_tfidf.py
-> Test set (test_210.csv) still SEALED.

C:\Users\Comsci and Crypto\Desktop\Sample\CF-2\ml_training\scripts>py 4b_tune_tfidf.py
Training samples loaded: 490  <- must be 490
Alpha fixed at         : 1.5  (from Step 4)
Test set               : still SEALED

Generating Figure 1: Stratified K-Fold split visualization...
  Saved -> C:\Users\Comsci and Crypto\Desktop\Sample\CF-2\ml_training\scripts\..\outputs\figures\fig1_stratified_kfold_split.png
========================================================================================
    max_feat   min_df      Acc%     ±Acc    Prec%     Rec%        F1
========================================================================================
  mf=5,000    md=1        81.22%  +-3.45%   81.30%   81.22%  0.8101
  mf=5,000    md=2        80.00%  +-3.14%   80.17%   79.96%  0.7975
  mf=5,000    md=3        80.61%  +-3.23%   81.05%   80.61%  0.8050
  mf=5,000    md=5        76.33%  +-5.41%   76.84%   76.32%  0.7644

  mf=10,000   md=1        80.20%  +-2.78%   80.34%   80.20%  0.7994
  mf=10,000   md=2        80.00%  +-3.14%   80.17%   79.96%  0.7975
  mf=10,000   md=3        80.61%  +-3.23%   81.05%   80.61%  0.8050
  mf=10,000   md=5        76.33%  +-5.41%   76.84%   76.32%  0.7644

  mf=15,000   md=1        79.80%  +-2.53%   79.92%   79.79%  0.7951
  mf=15,000   md=2        80.00%  +-3.14%   80.17%   79.96%  0.7975
  mf=15,000   md=3        80.61%  +-3.23%   81.05%   80.61%  0.8050
  mf=15,000   md=5        76.33%  +-5.41%   76.84%   76.32%  0.7644

  mf=20,000   md=1        79.80%  +-2.53%   79.92%   79.79%  0.7951
  mf=20,000   md=2        80.00%  +-3.14%   80.17%   79.96%  0.7975
  mf=20,000   md=3        80.61%  +-3.23%   81.05%   80.61%  0.8050
  mf=20,000   md=5        76.33%  +-5.41%   76.84%   76.32%  0.7644
========================================================================================

  Best config: max_features=5,000, min_df=1
  CV Acc : 81.22%  +-3.45%
  CV F1  : 0.8101
  vs Step 4 baseline: 80.00%  (↑ 1.22pp)

Generating Figure 2: Grid search heatmap...
  Saved -> C:\Users\Comsci and Crypto\Desktop\Sample\CF-2\ml_training\scripts\..\outputs\figures\fig2_gridsearch_heatmap.png
Generating Figure 3: Accuracy vs stability error bar plot...
  Saved -> C:\Users\Comsci and Crypto\Desktop\Sample\CF-2\ml_training\scripts\..\outputs\figures\fig3_accuracy_errorbar.png
Generating Figure 4: Per-fold accuracy of best configuration...
  Saved -> C:\Users\Comsci and Crypto\Desktop\Sample\CF-2\ml_training\scripts\..\outputs\figures\fig4_best_config_folds.png

=================================================================
  STEP 4b COMPLETE — ALL FIGURES SAVED
=================================================================
  fig1  Stratified K-Fold split    -> fold balance transparency
  fig2  Grid search heatmap        -> accuracy + std across all configs
  fig3  Accuracy vs stability      -> error bar + F1 overlay vs baseline
  fig4  Per-fold best config       -> fold-by-fold stability proof

  Best config found:
    BEST_ALPHA        = 1.5
    BEST_MAX_FEATURES = 5000
    BEST_MIN_DF       = 1
    CV Acc            = 81.22%  +-3.45%
    CV F1             = 0.8101

-> Open 5final_eval.py and set:
     BEST_ALPHA        = 1.5
     BEST_MAX_FEATURES = 5000
     BEST_MIN_DF       = 1
-> Then run: py 5final_eval.py  (unseals test_210.csv — one time only)
-> Test set (test_210.csv) still SEALED.

C:\Users\Comsci and Crypto\Desktop\Sample\CF-2\ml_training\scripts>py 5final_eval.py
Train : 490  <- must be 490
Test  : 210   <- must be 210

==========================================================
  CV RESULTS (training pool only, n=490, test still sealed)
==========================================================
  Fold       Accuracy   Precision   Recall       F1
  --------------------------------------------------
  Fold 1        84.69%      84.61%    84.63%  0.8460
  Fold 2        77.55%      77.97%    77.59%  0.7752
  Fold 3        85.71%      85.95%    85.73%  0.8566
  Fold 4        80.61%      80.59%    80.52%  0.8038
  Fold 5        77.55%      77.36%    77.62%  0.7687
  --------------------------------------------------
  Mean         81.22%      81.30%    81.22%  0.8101
  Std    +-    3.45%  +-   3.45%  +-  3.42%  +-0.0358

==========================================================
  FINAL HOLDOUT RESULTS  (test_210.csv — unsealed once)
==========================================================
  Alpha              : 1.5
  max_features       : 5000
  min_df             : 1

  Train accuracy     : 97.76%
  Holdout accuracy   : 84.76%
  Overfit gap        : 12.99%  (train - holdout)

  ── Macro averages (unweighted mean across 3 classes) ──
  Macro Precision    : 84.76%  (0.8476)
  Macro Recall       : 84.77%  (0.8477)
  Macro F1           : 84.73%  (0.8473)

  ── Weighted averages (weighted by class support) ──
  Weighted Precision : 84.75%  (0.8475)
  Weighted Recall    : 84.76%  (0.8476)
  Weighted F1        : 84.72%  (0.8472)

  ── Child safety metrics ──
  Bi-decision acc    : 91.43%  (Overstimulating vs Safe)
  Overstimulating recall : 90.00%  (missed detections = 6/70)
  CV mean (5-fold)   : 81.22% +-3.45%

==========================================================
  CLASSIFICATION REPORT (holdout, per-class breakdown)
==========================================================
                 precision    recall  f1-score   support

    Educational     0.8507    0.8261    0.8382        69
        Neutral     0.8406    0.8169    0.8286        71
Overstimulating     0.8514    0.9000    0.8750        70

       accuracy                         0.8476       210
      macro avg     0.8476    0.8477    0.8473       210
   weighted avg     0.8475    0.8476    0.8472       210


==========================================================
  PER-CLASS METRICS TABLE
==========================================================
  Class                 Precision   Recall       F1   Support
  ----------------------------------------------------------
  Educational              85.07%    82.61%    83.82%       69
  Neutral                  84.06%    81.69%    82.86%       71
  Overstimulating          85.14%    90.00%    87.50%       70

==========================================================
  CONFUSION MATRIX (rows=actual, columns=predicted)
==========================================================
                             Educa     Neutr     Overs
             Educational        57         7         5
                 Neutral         7        58         6
         Overstimulating         3         4        63

==========================================================
  THESIS SUMMARY ROW — Complement NB  (paste into Chapter 5)
==========================================================
  Holdout Acc : 84.76%
  CV Acc      : 81.22% +-3.45%
  Overfit Gap : 12.99%
  Macro P     : 0.8476
  Macro R     : 0.8477
  Macro F1    : 0.8473
  Bi-Dec Acc  : 91.43%
  Over. Recall: 90.00%

-> These are your FINAL numbers.
-> Do NOT re-run with different parameters after this.
-> Paste results into 6generate_figures.py

C:\Users\Comsci and Crypto\Desktop\Sample\CF-2\ml_training\scripts>
Microsoft Windows [Version 10.0.19045.6466]
(c) Microsoft Corporation. All rights reserved.

C:\Users\Comsci and Crypto\Desktop\Sample\CF-2\ml_training\scripts>py 6train_nb.py
═══ Training Final Thesis Model (NB-Only) ═══
Loaded 490 training samples.
Loaded 210 test samples.
Vectorizing text...
Training ComplementNB...

=======================================================
NAÏVE BAYES HOLDOUT ACCURACY: 83.81%
=======================================================

[Standalone NB] Detailed Classification Report:
                 precision    recall  f1-score   support

    Educational     0.8462    0.7971    0.8209        69
        Neutral     0.8406    0.8169    0.8286        71
Overstimulating     0.8289    0.9000    0.8630        70

       accuracy                         0.8381       210
      macro avg     0.8386    0.8380    0.8375       210
   weighted avg     0.8385    0.8381    0.8375       210

=======================================================

Success! New 'nb_model.pkl' saved to C:\Users\Comsci and Crypto\Desktop\Sample\CF-2\backend\app\models
Microsoft Windows [Version 10.0.19045.6466]
(c) Microsoft Corporation. All rights reserved.

C:\Users\Comsci and Crypto\Desktop\Sample\CF-2\ml_training\scripts>py 7test_hybridfusion.py

=================================================================
CHILDFOCUS — HYBRID EVALUATION  (v5 — 210 videos, config-aligned)
=================================================================
Source       : test_clean.csv — NB NEVER saw these 210 videos
Fusion       : v5-confidence-gated-aligned
BASE_ALPHA   : 0.6 (NB conf >= 0.4)
LOW_ALPHA    : 0.15 (NB conf <  0.4)
H_OVERRIDE   : Score_H < 0.07 → cannot be Overstimulating
Thresholds   : Block >= 0.3 | Allow <= 0.12
Cookies      : ✓ found
Sample       : 70 per class = 210 total

[HYBRID] Loaded 210 videos from test_clean.csv
[HYBRID]   Educational: 69
[HYBRID]   Neutral: 71
[HYBRID]   Overstimulating: 70
[HYBRID]   Selected 69 Educational videos
[HYBRID]   Selected 70 Neutral videos
[HYBRID]   Selected 70 Overstimulating videos
[HYBRID] Resuming — 14 already done.

[HYBRID] Videos to process: 195 (14 already done)


[ 15/210] video_id=MG10BPnR53k | class=Overstimulating
        title: 'Who Is Eating Sugar Secretly? 🍬🤫 #Shorts #KidsStory #FunLearning'

[SAMPLER] ══════════════════════════════════════
[SAMPLER] Analyzing: MG10BPnR53k
[SAMPLER] Found cookies.txt, using preemptively to bypass bot-blocks.
[SAMPLER] Download attempt 1: 6.3s
[SAMPLER] ✓ 'Who Is Eating Sugar Secretly? 🍬🤫 #Shorts #KidsStory #FunLearning' (22.4s)
[SAMPLER] Short video (22s) — ad-skip disabled
[SAMPLER] Segments: [('S1', 0), ('S2', 1), ('S3', 2)] | seg_dur=20s | ad_skip=0s
[SAMPLER] Thumbnail: 0.1823
[SAMPLER] Analysis: 19.2s
[SAMPLER] ✓ Score: 0.2086 → Neutral
[SAMPLER] ✓ Total runtime: 25.8s
[SAMPLER] ══════════════════════════════════════

[403/210] video_id=L8A4XbM5sXA | class=Educational
        title: 'Dora Becomes a Doctor! 🩺 | FULL EPISODE "Doctor Dora" | Dora the '

[SAMPLER] ══════════════════════════════════════
[SAMPLER] Analyzing: L8A4XbM5sXA
[SAMPLER] Found cookies.txt, using preemptively to bypass bot-blocks.
[SAMPLER] Download attempt 1: 41.8s
[SAMPLER] ✓ 'Dora Becomes a Doctor! 🩺 | FULL EPISODE "Doctor Dora" | Dora the Explorer' (1421.8s)
[SAMPLER] Ad-skip: first 15s excluded from analysis
[SAMPLER] Segments: [('S1', 15), ('S2', 29), ('S3', 43)] | seg_dur=20s | ad_skip=15s
[SAMPLER] Thumbnail: 0.4
[SAMPLER] Analysis: 1.4s
[SAMPLER] ✓ Score: 0.1103 → Safe
[SAMPLER] ✓ Total runtime: 43.2s
[SAMPLER] ══════════════════════════════════════

[YTAPI] ✓ Scraped 28 hidden keywords for L8A4XbM5sXA
  [NB] Tags: 28 + 28 scraped = 28
[NB] 'Dora Becomes a Doctor! 🩺 | FULL EPISODE "Doct' → Educational | Score_NB=0.245 | P(over)=0.245
  ✗ [    Educational →         Neutral] NB=0.245(conf=0.60) H=0.110 α=0.6 Final=0.191 [FULL] (46.48s) 'Dora Becomes a Doctor! 🩺 | FULL EPI'
        Elapsed: 116.6min | ETA: ~0min remaining

[HYBRID] All 209 videos processed.

=================================================================
HYBRID EVALUATION RESULTS  (v5 — 210-video test_clean.csv)
=================================================================
Total attempted  : 209
Valid results    : 209
Skipped / Error  : 0

Fusion v5-confidence-gated-aligned
BASE_ALPHA=0.6 (conf>=0.4) | LOW_ALPHA=0.15 | H_OVERRIDE=0.07
Block >= 0.3 | Allow <= 0.12

3-CLASS CLASSIFICATION REPORT:
                 precision    recall  f1-score   support

    Educational     0.7692    0.1449    0.2439        69
        Neutral     0.4882    0.8857    0.6294        70
Overstimulating     0.8406    0.8286    0.8345        70

       accuracy                         0.6220       209
      macro avg     0.6993    0.6197    0.5693       209
   weighted avg     0.6990    0.6220    0.5708       209

3-Class Confusion Matrix:
                             Educa         Neutr         Overs
         Educational            10            53             6
             Neutral             3            62             5
     Overstimulating             0            12            58

── 3-Class Summary ──
Overall Accuracy   : 0.6220  (62.20%)
Macro Precision    : 0.6993
Macro Recall       : 0.6197
Macro F1           : 0.5693

── Binary Summary (Safe vs Overstimulating) ──
                 precision    recall  f1-score   support

           Safe     0.8406    0.8286    0.8345        70
Overstimulating     0.9143    0.9209    0.9176       139

       accuracy                         0.8900       209
      macro avg     0.8774    0.8747    0.8760       209
   weighted avg     0.8896    0.8900    0.8898       209

Binary Confusion Matrix:
                                  Safe   Overstimulating
                Safe               128                11
     Overstimulating                12                58
Binary Accuracy    : 0.8900  (89.00%)

── Child Safety Metric ──
Overstimulating Recall : 0.8286 (82.86%)
Missed detections      : 11 out of 70

── Path Breakdown ──
Full video analysis  : 206
Thumbnail-only       : 3
H_OVERRIDE triggered : 8  (Score_H < 0.07)
LOW_ALPHA used       : 23  (NB conf < 0.4)

[HYBRID] Results JSON → C:\Users\Comsci and Crypto\Desktop\Sample\CF-2\ml_training\scripts\..\outputs\hybrid_210_results.json
[HYBRID] Report TXT  → C:\Users\Comsci and Crypto\Desktop\Sample\CF-2\ml_training\scripts\..\outputs\hybrid_210_report.txt

=================================================================
THESIS NUMBERS — PASTE INTO CHAPTER 5 SECTION 5.3.3
=================================================================
Source        : test_clean.csv (209 valid / 209 attempted)
Fusion        : v5-confidence-gated-aligned
3-class acc   : 0.6220 (62.20%)
Macro F1      : 0.5693
Binary acc    : 0.8900 (89.00%)
Overst. recall: 0.8286

Per-class breakdown:
  Educational       : P=0.7692  R=0.1449  F1=0.2439  n=69
  Neutral           : P=0.4882  R=0.8857  F1=0.6294  n=70
  Overstimulating   : P=0.8406  R=0.8286  F1=0.8345  n=70

3-Class Confusion Matrix:
                             Educa         Neutr         Overs
         Educational            10            53             6
             Neutral             3            62             5
     Overstimulating             0            12            58

Binary Confusion Matrix:
                                  Safe   Overstimulating
                Safe               128                11
     Overstimulating                12                58
=================================================================

[HYBRID] Complete.
C:\Users\Comsci and Crypto\Desktop\Sample\CF-2\ml_training\scripts>py 8post_test_hyperparameter.py

======================================================================
CHILDFOCUS — FINAL HYBRID CONFIGURATION SEARCH
======================================================================
[FINAL] Loaded 209 valid video results

======================================================================
CONFIGURATION COMPARISON SUMMARY
======================================================================

Configuration                    Alpha   Block   Allow        F1   Accuracy
---------------------------------------------------------------------------
Original (thesis)                  0.4   0.750   0.350    0.1810     0.3254
Recalibrated basic                 0.4   0.185   0.110    0.4577     0.5311
NB-dominant (0.7 alpha)            0.7   0.300   0.150    0.5760     0.6124
NB-dominant (0.8 alpha)            0.8   0.300   0.150    0.5717     0.6124
NB-dominant (0.9 alpha)            0.9   0.300   0.150    0.5607     0.6029

======================================================================
GRID SEARCH — ALPHA × THRESHOLD COMBINATIONS
======================================================================

   Alpha    Block    Allow         F1   Accuracy
--------------------------------------------------
     0.5    0.280    0.180     0.6108     0.6124 ← BEST
     0.4    0.250    0.170     0.6107     0.6172
     0.6    0.280    0.170     0.6011     0.6172
     0.4    0.250    0.160     0.6010     0.6124
     0.6    0.280    0.180     0.6008     0.6124
     0.4    0.250    0.180     0.6006     0.6077
     0.6    0.300    0.180     0.5997     0.6029
     0.6    0.300    0.170     0.5992     0.6077
     0.5    0.280    0.170     0.5977     0.6029
     0.6    0.320    0.180     0.5967     0.5933

======================================================================
BEST CONFIGURATION — DETAILED REPORT
======================================================================

Alpha (NB weight)    : 0.5  (50% NB / 50% Heuristic)
Block threshold      : >= 0.28  (Overstimulating)
Allow threshold      : <= 0.18  (Educational)
Neutral range        : 0.18 < score < 0.28

Per-video results:
      video_id             true             pred     NB      H   Final
  -----------------------------------------------------------------
  ✗  gBUuQwN-0TM          Neutral      Educational  0.155  0.198  0.1769  '🐭 Learn to Draw a Cute Mouse S'
  ✗  GJuO4MzYrtI      Educational          Neutral  0.212  0.170  0.1910  'Thank You Teachers! | SciShow '
  ✗  hNFppd7L0Bg  Overstimulating          Neutral  0.489  0.068  0.2786  'OMG CAPCUT HAS SMOOTH VELOCITY'
  ✓  aMGQtiQUdkA  Overstimulating  Overstimulating  0.673  0.183  0.4279  'Baby Shark - Best Kids Dance V'
  ✓  kDdg2M1_EuE      Educational      Educational  0.182  0.111  0.1469  'The Alphabet Is So Much Fun | '
  ✓  _fsjQXoqFtM          Neutral          Neutral  0.271  0.179  0.2251  'Precious Plant GOES TO THE BIN'
  ✗  yc0GQ0tL0FE      Educational          Neutral  0.167  0.202  0.1847  'ABC Song for Babies &amp; Kids'
  ✓  mXMofxtDPUQ      Educational      Educational  0.230  0.101  0.1654  'Days of the Week Song | The Si'
  ✗  PVFNfmoH6pg          Neutral      Educational  0.117  0.158  0.1375  'How To Draw A Cute Husky With '
  ✗  DvpQHdQFJdE          Neutral      Educational  0.139  0.153  0.1459  'Study vlog 📚 ˙✧˖° #study #stud'
  ✓  VVgu67WS0DI          Neutral          Neutral  0.182  0.203  0.1928  'The Seven Little Goats and The'
  ✓  RM485oUuOhg          Neutral          Neutral  0.244  0.154  0.1991  'Switzerland: The Land of Pure '
  ✓  J5hKPShTi3E          Neutral          Neutral  0.301  0.172  0.2366  '4 Ways To Get Organized'
  ✗  UiFjZcOrMO0      Educational  Overstimulating  0.537  0.212  0.3744  'Kaboochi | Dance Song For Kids'
  ✓  MG10BPnR53k  Overstimulating  Overstimulating  0.606  0.209  0.4072  'Who Is Eating Sugar Secretly? '
  ✗  R0vgSne0jE8  Overstimulating          Neutral  0.221  0.190  0.2054  'Shapes are EVERYWHERE #shorts'
  ✓  PyOmuhdibVM          Neutral          Neutral  0.220  0.153  0.1862  'Nightly Routine #cleaning #apa'
  ✗  lWhqORImND0      Educational          Neutral  0.364  0.160  0.2621  'Old MacDonald Had A Farm + Mor'
  ✗  0W86K1jBJFI          Neutral      Educational  0.187  0.157  0.1719  'Little Red Riding Hood - Anima'
  ✓  2bUTxSPyCzo  Overstimulating  Overstimulating  0.682  0.215  0.4483  'Hide and Seek in One Color wit'
  ✗  UpDPJICdhhg      Educational          Neutral  0.418  0.001  0.2095  '6.6 LogicLike - Educational ga'
  ✓  q2PglyGrHqc  Overstimulating  Overstimulating  0.872  0.088  0.4799  'Baby Shark doo doo doo #shorts'
  ✓  0LnhNC_wHY8  Overstimulating  Overstimulating  0.436  0.200  0.3180  'Rain, Rain, Go Away Nursery Rh'
  ✓  _OcW8ZHGZr8      Educational      Educational  0.041  0.142  0.0914  'Back To School Reading for Kid'
  ✗  G8nuZu0VDp4      Educational          Neutral  0.243  0.176  0.2093  '5 Minute Stories Book Read-Alo'
  ✗  LDD6hX0lpj8          Neutral      Educational  0.152  0.137  0.1447  'aesthetic morning routine 🌸 #a'
  ✓  viurF6fz9Xg  Overstimulating  Overstimulating  0.443  0.194  0.3180  '$1 VS $1,000,000 99 NIGHTS IN '
  ✓  49df3lEIYyg  Overstimulating  Overstimulating  0.468  0.191  0.3294  'Baby Got a Boo Boo | Nursery R'
  ✓  6QdwB7ZCyMA      Educational      Educational  0.021  0.135  0.0779  '@Numberblocks- Meet Number Nin'
  ✗  daL2STCfFHw  Overstimulating          Neutral  0.343  0.215  0.2793  'Building Our Family Island in '
  ✓  xoamCzuQjy0      Educational      Educational  0.220  0.094  0.1565  'Colors Vocabulary Chant for Ki'
  ✗  c_NGdBIlu0M  Overstimulating          Neutral  0.260  0.144  0.2021  'aayu and pihu show 👦😍👩#shortvi'
  ✓  dmVNCtbcTdU  Overstimulating  Overstimulating  0.595  0.186  0.3904  'Hot Wheels Monster Trucks 1:6 '
  ✗  l_SSeAMwCDw          Neutral      Educational  0.122  0.151  0.1365  'Very Easy! Fish Drawing For Ki'
  ✓  tisxrc_6zbE  Overstimulating  Overstimulating  0.680  0.234  0.4571  'Chris collects toys for his ga'
  ✓  qJIYkbAgR8c  Overstimulating  Overstimulating  0.465  0.183  0.3239  'Yes Yes Vegetables | 🥕 The Hun'
  ✗  bt3ttH11QT8  Overstimulating          Neutral  0.338  0.159  0.2483  'Smashing a 34,000 Brick Statue'
  ✓  XOnHtStmbCI          Neutral          Neutral  0.290  0.200  0.2451  'Mickey and Donald Have a Farm '
  ✓  6M3IrkyNNT8      Educational      Educational  0.022  0.167  0.0944  '@Numberblocks- Meet Number Fiv'
  ✗  LSfqhUKf0X0  Overstimulating          Neutral  0.321  0.207  0.2639  'Villains are taking over! We n'
  ✓  FXjC69MmcPY      Educational      Educational  0.183  0.143  0.1633  '🌈🎵 Learn, Sing, and Explore! P'
  ✓  gB1dNEkogYk      Educational      Educational  0.245  0.056  0.1505  'Certified BOP'
  ✓  3JGOK5q5IMA          Neutral          Neutral  0.298  0.217  0.2578  '3-Ingredient Nutella Cookies! '
  ✓  0j_AbvBOWZw  Overstimulating  Overstimulating  0.530  0.116  0.3233  'More Baby Seals for You (Part3'
  ✗  s0bS-SBAgJI          Neutral      Educational  0.199  0.100  0.1496  'The Water Cycle- How rain is f'
  ✗  hkz51Td9k5A          Neutral      Educational  0.158  0.178  0.1683  "Read Aloud Kids Book: It's A F"
  ✗  sOxloXyOAKA          Neutral      Educational  0.247  0.067  0.1569  'Nature View Cristal beach Beut'
  ✓  gDInVjBUGC0      Educational      Educational  0.132  0.057  0.0941  'ELS: Phase 3 pronunciation'
  ✓  AGYPvz9Dov8          Neutral          Neutral  0.220  0.174  0.1968  'HIPPO VIDEOS FOR KIDS - Facts '
  ✗  r2bW4ySQr-4          Neutral      Educational  0.228  0.065  0.1465  'Creative Art For Kids Chair 🪑🌈'
  ✗  xw4BPniE2m4      Educational          Neutral  0.339  0.163  0.2512  'This Ruthless Spider Is the Ma'
  ✓  fhPaQwEoGGM  Overstimulating  Overstimulating  0.782  0.192  0.4869  'Kids Trapped in Minecraft!'
  ✗  LpUzg3Dc-m0      Educational          Neutral  0.309  0.193  0.2510  'Never Take Seashells From Beac'
  ✗  rbVkRDOjMa8  Overstimulating          Neutral  0.344  0.189  0.2667  'Sunny Bunnies | Shiny Bright B'
  ✗  qaRYVwND3SQ          Neutral      Educational  0.166  0.169  0.1673  'How to Blow Your Nose - Life S'
  ✓  89QRrnnYPNw          Neutral          Neutral  0.282  0.205  0.2438  'How Do Plants Grow? | Knowsy N'
  ✗  fKigjAnJrjg      Educational          Neutral  0.255  0.191  0.2230  'The Magic Magnet | Funny Stori'
  ✗  xfLYZeLV3go      Educational          Neutral  0.244  0.183  0.2133  'The Egg&#39;s Adventure | Educ'
  ✗  nk6QEJkNWHc          Neutral      Educational  0.163  0.150  0.1567  'A day in my cottage life'
  ✗  GKLRkTA64N0      Educational          Neutral  0.284  0.094  0.1891  'Numbers Silly Song'
  ✓  mepSmUC6HWM  Overstimulating  Overstimulating  0.682  0.222  0.4518  'Monsters Play Trash Basketball'
  ✓  zdQfvoKNLhE  Overstimulating  Overstimulating  0.482  0.230  0.3561  'Oh! Hero Piggy Saved the Littl'
  ✗  rtAwdYTjNhY      Educational          Neutral  0.293  0.131  0.2116  "Quick Quiz! Who's Larger? Male"
  ✓  C4l7rfjX49I          Neutral          Neutral  0.414  0.098  0.2561  'Who won? #shorts #family'
  ✗  k14YDC1Kn-o          Neutral      Educational  0.174  0.177  0.1758  'The Lion and The Mouse | Fairy'
  ✓  IzkmfRTktWc      Educational      Educational  0.219  0.087  0.1529  'Building Squares from Odd Numb'
  ✓  I_4Xw57Lt-k  Overstimulating  Overstimulating  0.783  0.184  0.4835  'Kids Cars Adventures Compilati'
  ✗  OtalQpNt2UQ      Educational          Neutral  0.277  0.183  0.2298  'Dora & Boots Go On a Puppy Adv'
  ✓  aBAOUEVdNJA          Neutral          Neutral  0.315  0.151  0.2330  'those school days... 😭🌷 #short'
  ✓  nkqet4ca25A          Neutral          Neutral  0.330  0.154  0.2419  'REALITY vs SLOW MO! 🎥'
  ✗  7SWvlUd2at8          Neutral      Educational  0.107  0.087  0.0968  '10 Easy Animal Drawings for Ki'
  ✓  VdzzE20zQC8      Educational      Educational  0.205  0.142  0.1734  'The Shapes Song | Nursery Rhym'
  ✓  q795N2khTXk  Overstimulating  Overstimulating  0.520  0.132  0.3260  'Would You Let Your Kids Play T'
  ✗  Rw3NxD-p8_U  Overstimulating          Neutral  0.349  0.179  0.2640  'Possible 1x1x1x1?? recent "hac'
  ✗  yTXtTX3t8fM      Educational  Overstimulating  0.418  0.147  0.2823  '9.1 LogicLike - Educational ga'
  ✓  qRDSgx5PF9g          Neutral          Neutral  0.315  0.175  0.2449  'Pictures that relieve anxiety!'
  ✗  R_CD6ellkyk          Neutral      Educational  0.259  0.056  0.1579  'Relaxing Video of a Sunrise Be'
  ✗  QsoRCSnhsQo      Educational          Neutral  0.228  0.153  0.1906  'Return Little Crocodile Home |'
  ✓  MvXOhpSuTx4  Overstimulating  Overstimulating  0.628  0.196  0.4121  'Shoo Be Doo Birthday🥳🎈# birthda'
  ✓  9w9GlT4Zyl8          Neutral          Neutral  0.375  0.180  0.2774  'Wall&#39;s 3 in 1 Ice Cream 🍦 '
  ✓  AS2gsXyn4ts  Overstimulating  Overstimulating  0.450  0.174  0.3125  "A Cute Zombie's Message Just f"
  ✓  BfodhLizt_A  Overstimulating  Overstimulating  0.712  0.189  0.4506  'Unboxing Disney Cars Tomica Ra'
  ✓  0JDCViWGn-0          Neutral          Neutral  0.293  0.183  0.2383  'Human Body Systems Overview (U'
  ✓  -PxShNDmIHw      Educational      Educational  0.043  0.083  0.0626  "Let's Spell Words ending in 'a"
  ✓  _f1QbJm1irY          Neutral          Neutral  0.196  0.167  0.1816  'The Smartest Giant in Town - A'
  ✓  Ae4MadKPJC0          Neutral          Neutral  0.247  0.127  0.1868  'Human Body 101 | National Geog'
  ✗  36n93jvjkDs      Educational          Neutral  0.236  0.193  0.2142  'Days of The Week Song For Kids'
  ✓  4DoO_Td6k8M  Overstimulating  Overstimulating  0.700  0.225  0.4625  'Behind Story of Pinkfong Baby '
  ✓  _Dwbf5WjjJA  Overstimulating  Overstimulating  0.396  0.179  0.2873  'IMPOSTERS are PRANKING US! FUL'
  ✓  -G49u2y-0U4          Neutral          Neutral  0.357  0.084  0.2205  'Mom and Dad use girls as mops '
  ✓  lmRmk2HrqYA          Neutral          Neutral  0.386  0.001  0.1937  'Animal Showdown 🐻🐊🦅 | 25 Minut'
  ✓  JhH-_QXzQ3o  Overstimulating  Overstimulating  0.467  0.118  0.2921  'Baby Seal: Not Home Alone (Ful'
  ✓  v3BCkHGnKT4  Overstimulating  Overstimulating  0.592  0.187  0.3892  'Andrea Becomes Superheroes to '
  ✓  2K4o11uxhs0      Educational      Educational  0.087  0.183  0.1350  'ABC Phonics Song + More Nurser'
  ✗  bWAIgN_DTxo      Educational          Neutral  0.283  0.091  0.1867  'Everything You Need to Know Ab'
  ✓  MmgFk1oL9a8          Neutral          Neutral  0.277  0.084  0.1805  '🥹how beautiful is this � ##viral'
  ✓  K0GujQE6Jqc      Educational      Educational  0.142  0.167  0.1547  'ABC Phonics Song - A for Apple'
  ✓  CUBciV4iPzs          Neutral          Neutral  0.374  0.175  0.2743  'Baking With Blippi | Food Vide'
  ✓  U9Q7Y3t4m3g      Educational      Educational  0.214  0.121  0.1673  'Good Morning Song For Children'
  ✗  0GAhHaK4y58      Educational          Neutral  0.304  0.170  0.2370  'Blippi Meets TALKING PLANETS! '
  ✗  mDzIA3VUFKI  Overstimulating          Neutral  0.388  0.151  0.2693  'Lunar New Year Dance 💃👯🏮 Talki'
  ✓  XSun4GUyhuY      Educational      Educational  0.105  0.222  0.1634  'Preschool Learning activities '
  ✓  QK_4HL1vK0o  Overstimulating  Overstimulating  0.655  0.225  0.4400  'Escaping 100 Layers of CANDY C'
  ✗  3CfvBCAPZyE      Educational          Neutral  0.302  0.182  0.2424  'Ms. Rachel Counts to 10 with B'
  ✗  tSEhQO0zlGQ          Neutral  Overstimulating  0.564  0.184  0.3743  'Thomas the Rescue Engine | Car'
  ✓  Mbc29yUtIxs          Neutral          Neutral  0.302  0.206  0.2538  'Microwave Meals: Lava Cake!'
  ✓  1szs8RMlhN8          Neutral          Neutral  0.331  0.174  0.2527  '1 carrot with 1 egg! your kids'
  ✗  y8kneKeNyCI          Neutral  Overstimulating  0.508  0.189  0.3483  'Belly Button Song Dance! Learn'
  ✓  NQOD8oBfZjs          Neutral          Neutral  0.218  0.168  0.1930  'A Day In The Life Of Serengeti'
  ✓  g2jdZ46nK-M      Educational      Educational  0.193  0.156  0.1742  'Shapes Song For Kids-Circle, T'
  ✓  fTYhA_j6cUw  Overstimulating  Overstimulating  0.783  0.147  0.4649  'Water Safety Song | Run Away f'
  ✗  X0SpQyIHVXQ      Educational          Neutral  0.312  0.171  0.2415  'What is in the fridge? 🐧'
  ✓  zTmjQz198f0  Overstimulating  Overstimulating  0.567  0.155  0.3611  'Fish 123 | Ten Little Fish | L'
  ✓  YSA2f19ABPk  Overstimulating  Overstimulating  0.678  0.194  0.4359  'Chris and friends in Magic Esc'
  ✓  aJxxbI9HIRU  Overstimulating  Overstimulating  0.676  0.208  0.4424  'Baby Shark - Sing and Dance - '
  ✓  bjRpNYq8Ts0          Neutral          Neutral  0.246  0.156  0.2012  'Exploring the Adorable World o'
  ✓  THYU9WgZD_Y  Overstimulating  Overstimulating  0.357  0.210  0.2834  'This is going to be AWESOME! #'
  ✓  hV5eWushhbQ  Overstimulating  Overstimulating  0.428  0.133  0.2807  '3 Skeletons Were Riding On A B'
  ✗  ao2IP7f23-8          Neutral      Educational  0.125  0.135  0.1299  'How To Draw A Hibiscus Flower '
  ✗  vInTXlOIRzQ          Neutral      Educational  0.130  0.101  0.1153  'How To Draw Cinnamoroll From H'
  ✓  RjpRL6tv3Cw          Neutral          Neutral  0.388  0.111  0.2499  'Bluey and Bingo Pretend to be '
  ✓  KGD1TJ_XjbI          Neutral          Neutral  0.321  0.203  0.2621  'Korean Cheesy Potato Pancake ('
  ✗  WhhPGrWxWsI      Educational          Neutral  0.216  0.201  0.2082  'Harbour Learning | Sea Animal '
  ✓  4HGwX5b44Yc  Overstimulating  Overstimulating  0.693  0.212  0.4526  'Superheroes Teamwork Adventure'
  ✗  SNF8b7KKJ2I          Neutral      Educational  0.181  0.127  0.1538  'Ecosystems for Kids | Plants, '
  ✓  oY-H2WGThc8          Neutral          Neutral  0.313  0.170  0.2417  'Clean Up Song for Children - b'
  ✓  qCs0pm_D_hM  Overstimulating  Overstimulating  0.588  0.167  0.3775  'Colors for Children | Toy Trai'
  ✓  IPr_Ay-7aT0      Educational      Educational  0.201  0.101  0.1510  'The Phonics Song + More Presch'
  ✗  7aKsPwH1iss          Neutral      Educational  0.225  0.073  0.1492  'beautiful sunset video #shorts'
  ✓  1nJ5VbyD6tY      Educational      Educational  0.180  0.153  0.1667  '2D Shapes Song'
  ✗  qqwRkhwrJek          Neutral  Overstimulating  0.381  0.203  0.2920  'Jingle Bells,  Christmas Song,'
  ✓  D5XMzXizUKA  Overstimulating  Overstimulating  0.856  0.171  0.5135  '60 Minutes Ultimate Cooking To'
  ✓  UvVKtqlxJq8  Overstimulating  Overstimulating  0.440  0.187  0.3138  'Try NOT To Laugh: Silly Jokes '
  ✓  xgJjg4HSo4A          Neutral          Neutral  0.256  0.178  0.2169  'Organize – Teach Kids to Stay '
  ✓  PL7vHFGRViA  Overstimulating  Overstimulating  0.775  0.188  0.4817  'Satisfying with Unboxing & Rev'
  ✗  2jMyZ0OZan8          Neutral      Educational  0.113  0.164  0.1382  'How To Draw The Cutest Fish Us'
  ✓  zZz91aicTuU  Overstimulating  Overstimulating  0.497  0.179  0.3383  'Power Rangers Ninja Kidz Battl'
  ✓  WkbfeCays4w  Overstimulating  Overstimulating  0.765  0.192  0.4781  'Baby Shark Sing and Dance Song'
  ✗  0n_J2z-ILXo      Educational          Neutral  0.385  0.143  0.2643  'Humpty Dumpty | + More Kids So'
  ✓  geyzFEhOu_s  Overstimulating  Overstimulating  0.557  0.207  0.3816  'Rich Princess Treats the Princ'
  ✗  NkY57mizvMI      Educational          Neutral  0.224  0.167  0.1955  '🐯 Meet TINKER! 🐯 | Your Favori'
  ✗  b2sjCyAsFbM      Educational          Neutral  0.270  0.163  0.2167  'Peter Rabbit - Camping by the '
  ✓  vXkY5wIWu18  Overstimulating  Overstimulating  0.544  0.177  0.3609  'Kaden Eva & Emma Jump in a Hol'
  ✓  9M4JWcKEdL0      Educational      Educational  0.262  0.092  0.1770  'Three Space Weather Missions L'
  ✓  cixocSycC3A  Overstimulating  Overstimulating  0.475  0.153  0.3140  'So Satisfying 😳'
  ✓  eg2mr1IticM  Overstimulating  Overstimulating  0.537  0.158  0.3473  'EASTER EGG ASMR! 🐣💚🩷🩵   #asmr #a'
  ✓  e2JMt_YJveM          Neutral          Neutral  0.231  0.156  0.1932  'What are clouds? ☁☁ How are th'
  ✓  1y8MZvPhCKE      Educational      Educational  0.042  0.123  0.0824  'The Best of Alphablock O  | Le'
  ✓  Kx241A5WsDI  Overstimulating  Overstimulating  0.584  0.189  0.3869  'Blippi & Yellowcard GO GO GO o'
  ✗  eclBbnyOsYA          Neutral  Overstimulating  0.469  0.165  0.3167  'Learn Letters! - ABC Undersea '
  ✓  ki5wRkJWPP8  Overstimulating  Overstimulating  0.697  0.201  0.4486  'Vlad plays with magic masks'
  ✗  CGSJCE0Rqnw      Educational          Neutral  0.293  0.178  0.2352  'Is laughter ACTUALLY the best '
  ✓  pr5U8x-kCMM  Overstimulating  Overstimulating  0.421  0.166  0.2934  'LARVA - RAINING | Larva 2017 |'
  ✗  ozsgl_sLnHQ      Educational          Neutral  0.168  0.224  0.1961  'Learn About Emotions and Feeli'
  ✓  qoXM4iykEz8          Neutral          Neutral  0.208  0.178  0.1933  "How to Get a Good Night's Slee"
  ✗  gbsgbcd_-2U          Neutral  Overstimulating  0.417  0.169  0.2932  'PAW Patrol Rescue Wheels Adven'
  ✓  -gajDGMq7Lg  Overstimulating  Overstimulating  0.405  0.186  0.2954  'When a girl gets you angry 😂 #'
  ✗  Oyu80bhbrNk      Educational          Neutral  0.249  0.142  0.1955  'Screen Time That Earns Its Pla'
  ✓  KyBYuEgvFl0  Overstimulating  Overstimulating  0.764  0.167  0.4657  'Baby Car | Car Songs | Pinkfon'
  ✓  mJaxCjNJDww      Educational      Educational  0.100  0.161  0.1304  'Learning Videos for Toddlers |'
  ✓  KsaGjQDxjJQ  Overstimulating  Overstimulating  0.734  0.215  0.4748  '30 Minutes Satisfying with Unb'
  ✓  ws_D5nXOAJg          Neutral          Neutral  0.247  0.128  0.1875  'The Stunning Life Cycle Of A L'
  ✓  Ov-KDfyUR0A  Overstimulating  Overstimulating  0.608  0.235  0.4217  'Vlad and Niki build a construc'
  ✗  CYEyoUUfA8A      Educational          Neutral  0.248  0.128  0.1880  'Routines That Build More Than '
  ✓  FjtyU_KNits      Educational      Educational  0.103  0.160  0.1310  'Learn the Alphabet Sounds with'
  ✗  _C-cxckDJX0          Neutral      Educational  0.159  0.140  0.1497  'morning routine #shorts'
  ✓  r1Hbzt_tjGg          Neutral          Neutral  0.283  0.179  0.2310  'Cute Rainbow Painting Hack for'
  ✓  fdm1_QInzfM          Neutral          Neutral  0.179  0.188  0.1832  'How To Draw An Emoji Folding S'
  ✗  ZY-NF22bKXY          Neutral      Educational  0.131  0.182  0.1564  'Aesthetic Quiet Morning Study '
  ✓  Lvkf2bwch7w          Neutral          Neutral  0.247  0.137  0.1920  'My morning routine #shorts'
  ✓  gV0w_wrU750  Overstimulating  Overstimulating  0.583  0.175  0.3793  '&quot;Baby Shark&quot; - The P'
  ✓  10OJhIc_-vw  Overstimulating  Overstimulating  0.459  0.184  0.3219  'Minecraft Speedrunner VS 6 Hun'
  ✓  23C5twKp72Q      Educational      Educational  0.198  0.106  0.1522  "Meet the Shapes | 'Rectangle' "
  ✗  Me9Ex4QZnbU      Educational  Overstimulating  0.595  0.192  0.3936  'Wheels On The Bus story with A'
  ✗  EoRXcJRtKq4          Neutral      Educational  0.149  0.173  0.1607  'How to draw a cute girl easy |'
  ✓  zHT2nY-Gq1I  Overstimulating  Overstimulating  0.550  0.212  0.3810  'Kids Learn About All Their Fee'
  ✗  r4MfjpBQCvQ      Educational  Overstimulating  0.407  0.153  0.2801  'The Shark is Coming+More | Yum'
  ✓  l5RrO6cYy_A          Neutral          Neutral  0.205  0.185  0.1952  'day in my life as a college st'
  ✓  zugFjQ8tN3E  Overstimulating  Overstimulating  0.440  0.187  0.3130  'Eva Emma & Kaden Playground Ge'
  ✓  yeVTvh9HU8s  Overstimulating  Overstimulating  0.447  0.142  0.2941  'Blippi Meets Dr. Ted the Puppy'
  ✗  3ZHYQ6f1BhU      Educational          Neutral  0.234  0.214  0.2240  'Cavities - The Dr. Binocs Show'
  ✗  quUlfkOyH0g          Neutral      Educational  0.232  0.103  0.1676  'Duck Drawing | Drawing for kid'
  ✗  pfRuLS-Vnjs      Educational          Neutral  0.256  0.113  0.1844  'The Shapes Song'
  ✗  s_hAnPdkBTI          Neutral      Educational  0.182  0.132  0.1573  'How to wake up in the morning '
  ✓  MfIwF6QoRm4  Overstimulating  Overstimulating  0.835  0.189  0.5117  'Police Officer Visits Doctors '
  ✓  Nx9rbyz3iDc          Neutral          Neutral  0.342  0.185  0.2636  'What really happens when we co'
  ✓  0VLxWIHRD4E      Educational      Educational  0.217  0.075  0.1463  "Let's Count to 20 Song For Kid"
  ✗  DS_PQYIGrTc          Neutral      Educational  0.183  0.085  0.1340  'How To Draw The Pusheen Cat'
  ✗  hCaHFv2YWn8  Overstimulating          Neutral  0.324  0.082  0.2029  'Lincoln Loud&#39;s Reaction! 😬'
  ✓  7wNyw7DlY0A          Neutral          Neutral  0.231  0.144  0.1880  'How to study when you are tire'
  ✓  jiEv6VTDt5c      Educational      Educational  0.093  0.132  0.1125  'Learn to Read - 3-Letter Word '
  ✗  r8iGnwD8i9I      Educational  Overstimulating  0.470  0.173  0.3216  'ABC Song and Many More Nursery'
  ✓  jOGuabdHYaE  Overstimulating  Overstimulating  0.823  0.153  0.4881  'Baby Shark Dance 2 | Sing and '
  ✗  g6CVApMxmqg      Educational          Neutral  0.237  0.190  0.2138  'Phonics, Counting, Colors + Mo'
  ✓  d2S87jXhlV0  Overstimulating  Overstimulating  0.737  0.186  0.4613  'Original Baby Shark | Go #Baby'
  ✓  v7cuoaoUTKQ  Overstimulating  Overstimulating  0.857  0.216  0.5365  'Baby Shark Dance | Pinkfong Si'
  ✗  KoDg_L6B2y4      Educational          Neutral  0.242  0.214  0.2283  'Let Them Cook: Collard Greens '
  ✓  Be4j6lJpFTQ      Educational      Educational  0.105  0.177  0.1409  'Counting to 20 Song For Kids |'
  ✗  kopx0X8XTMM      Educational          Neutral  0.271  0.146  0.2088  'Stand like a flamingo with Ms.'
  ✓  9h8RmsptqaI  Overstimulating  Overstimulating  0.412  0.183  0.2977  'New Chicken Dance #shorts  by '
  ✗  rifuEZFieaI      Educational          Neutral  0.189  0.212  0.2009  'SUSTAINABLE DEVELOPMENT |#clim'
  ✓  RxCwVQGlDis  Overstimulating  Overstimulating  0.781  0.103  0.4423  'Baby Shark Dance | DJ Raphi So'
  ✓  6VQip5MI5d8  Overstimulating  Overstimulating  0.808  0.173  0.4907  'Johny Johny Yes Papa + Wheels '
  ✓  LMuBxwVy3HM      Educational      Educational  0.221  0.132  0.1766  'Peppa Pig Full Episodes | Pott'
  ✗  qbIgrxIfHxI          Neutral      Educational  0.140  0.166  0.1530  'Beautiful Nature with Rural Li'
  ✗  e87xaEjYMtg      Educational          Neutral  0.261  0.129  0.1948  'Peanut Butter And Jelly �  🍇 #s'
  ✓  YI0bd8Tw8rE      Educational      Educational  0.026  0.098  0.0624  '@Numberblocks- Meet Number Sev'
  ✗  HrDl_1Ov8gc      Educational  Overstimulating  0.427  0.199  0.3133  'Learn Colors with Wonderville '
  ✓  L8A4XbM5sXA      Educational      Educational  0.245  0.110  0.1777  'Dora Becomes a Doctor! 🩺  | FUL'

Classification Report:
                 precision    recall  f1-score   support

    Educational     0.5263    0.4348    0.4762        69
        Neutral     0.4691    0.5429    0.5033        70
Overstimulating     0.8451    0.8571    0.8511        70

       accuracy                         0.6124       209
      macro avg     0.6135    0.6116    0.6102       209
   weighted avg     0.6139    0.6124    0.6108       209

Confusion Matrix:
                           Educa       Neutr       Overs
         Educational          30          33           6
             Neutral          27          38           5
     Overstimulating           0          10          60

==================================================
FINAL METRICS (Best Config)
==================================================
Accuracy           : 0.6124  (61.24%)
Weighted Precision : 0.6139
Weighted Recall    : 0.6124
Weighted F1-Score  : 0.6108

Per-class:
  Educational       : P=0.5263  R=0.4348  F1=0.4762  n=69
  Neutral           : P=0.4691  R=0.5429  F1=0.5033  n=70
  Overstimulating   : P=0.8451  R=0.8571  F1=0.8511  n=70

[FINAL] Report saved → C:\Users\Comsci and Crypto\Desktop\Sample\CF-2\ml_training\scripts\..\outputs\post-hyperparameter_report.txt

======================================================================
PASTE THESE INTO YOUR THESIS — CHAPTER 5
======================================================================
Fusion config  : alpha=0.5 (NB), beta=0.5 (Heuristic)
Thresholds     : Block >= 0.28, Allow <= 0.18
Accuracy       : 0.6124 (61.24%)
Precision      : 0.6139
Recall         : 0.6124
F1-Score       : 0.6108

Per-class breakdown:
  Educational       : P=0.5263  R=0.4348  F1=0.4762
  Neutral           : P=0.4691  R=0.5429  F1=0.5033
  Overstimulating   : P=0.8451  R=0.8571  F1=0.8511

Confusion Matrix:
                           Educa       Neutr       Overs
         Educational          30          33           6
             Neutral          27          38           5
     Overstimulating           0          10          60
======================================================================